In [1]:
import concurrent.futures
import tempfile
import time
from pathlib import Path
import os
import io
import html as html_lib
import json
import base64
import zipfile
import unicodedata
import warnings
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from plotly.offline import get_plotlyjs
from datetime import datetime, timedelta, timezone
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px


warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# @param ["24h", "7d", "15d", "30d", "60d", "365d"]
PERIODO_ATUAL = "30d"
PERIOD_DAYS_MAP = {'24h': 1, '7d': 7, '15d': 15, '30d': 30, '60d': 60, '365d': 365}

dados_coletados = {}
cards_html = []
fallback_indicadores = []
falhas_indicadores = []

SESSION = requests.Session()
_retry_cfg = Retry(
    total=2,
    backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
SESSION.mount('https://', HTTPAdapter(max_retries=_retry_cfg, pool_connections=8, pool_maxsize=8))
SESSION.mount('http://', HTTPAdapter(max_retries=_retry_cfg, pool_connections=8, pool_maxsize=8))
SESSION.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
# Na função get_ons_resource, mude o timeout de download:
print("✅ Célula 1: Bibliotecas, sessão HTTP com retry e período configurados.")


✅ Célula 1: Bibliotecas, sessão HTTP com retry e período configurados.


In [2]:
# Slugs verificados diretamente contra o catálogo oficial do CKAN do ONS
# (https://dados.ons.org.br/dataset/?organization=ons). Comentário indica quando
# não existe um dataset equivalente exato e foi usado o mais próximo disponível.
datasets_selecionados = {
    'Armazenamento por Subsistema':   'ear-diario-por-subsistema',
    'ENA por Subsistema':             'ena-diario-por-subsistema',
    'Carga Energia Diária':       'carga-energia',
    'CMO Semanal':                'cmo-semanal',
    'CMO Semi-Horário':           'cmo-semi-horario',
    'Intercâmbio Nacional':       'intercambio-nacional',
    'Constrained off Solar':      'restricao_coff_fotovoltaica',
    'EAR por Bacia':              'ear-diario-por-bacia',
    'EAR por Equivalente de Energia':'ear-diario-por-ree-reservatorio-equivalente-de-energia',
    'EAR por Reservatório':       'ear-diario-por-reservatorio',
    'ENA por Equivalente de Energia': 'ena-diario-por-ree-reservatorio-equivalente-de-energia',
    'ENA por Reservatório':       'ena-diario-por-reservatorio',
    'Dados Hidrológicos':         'dados-hidrologicos-res',
    'CVU Usina Térmica':          'cvu-usitermica',
    'PLD CCEE':                   'ccee-pld-media-diaria',
    'Previsão Eólica':            'programacao_x_previsao',
    'Previsão Solar':             'programacao_x_previsao',
    'Carga de Energia Programada':           'carga-energia-programada',
    'Demanda Máxima Diária por Subsistema':             'demanda_maxima_di',
    'Taxas TEIF & TEIP':   'taxa_teif_teip',
    'Dados dos Programados dos Elementos de Fluxo Controlado':      'programacao_fluxo_controlado',
    'Atendimento aos Limites Sistêmicos':    'ind_confiarb_atls',
    'Desempenho da Frequência em Regime Permanente':           'ind_qualid_dfp_regime',
    'Interrupções de Carga':      'interrupcao_carga',
    'Linhas de Transmissão Rede Operação':            'linha-transmissao',
    'Controle de Carregamento de Transformadores': 'ind_confiarb_ccat',
    'Subestação de Rede de Operação':      'subestacao',
    'Capacidade de Transformação da Rede Básica':   'capacidade-transformacao',
    'Controle de Carregamento de Linhas de Transmissão':      'ind_confiarb_ccal',
    'Carga de Energia Mensal':         'carga-mensal',
    'Intercâmbios entre Subsistemas':  'intercambio-nacional',
}

# ── Datasets pesados — descomente individualmente se precisar ────────────────
datasets_pesados = {
    # 'Constrained-off Eólica': 'restricao_coff_eolica_usi',   # ~197s
    # 'Geração Programada':     'programacao_diaria',           # ~169s
    # 'Geração por Usina':      'geracao-usina-2',              # ~160s
    # 'Energia Vertida':        'energia-vertida-turbinavel',   # ~140s
    # 'Fator de Capacidade':    'fator-capacidade-2',           # ~135s
    # 'Disponibilidade Usinas': 'disponibilidade_usina',        # ~110s
}
# Para carregar os pesados junto, descomente a linha abaixo:
# datasets_selecionados.update(datasets_pesados)
PALETAS = {
    'default': ['#0078D4', '#E63946', '#2A9D8F', '#F4A261', '#9b5de5', '#00b4d8', '#fb8500', '#7209b7'],
    'hidrologia': ['#00B4D8', '#48CAE4', '#90E0EF', '#0077B6', '#0096C7', '#ADE8F4', '#00A6FB'],
    'carga': ['#2A9D8F', '#E76F51', '#F4A261', '#277DA1', '#E9C46A', '#F8961E', '#D62828'],
    'mercado': ['#9B5DE5', '#4361EE', '#4CC9F0', '#F72585', '#B5179E', '#7B2CBF', '#560BAD'],
    'transmissao': ['#D62828', '#F77F00', '#FCBF49', '#EAE2B7', '#003049', '#669BBC', '#C1121F'],
    'armazenamento': ['#0078D4', '#2A9D8F', '#E76F51', '#9b5de5', '#f4a261', '#e76f51'],
    'ena': ['#00B4D8', '#48CAE4', '#0096C7', '#0077B6', '#90E0EF', '#ADE8F4'],
    'cmo': ['#9B5DE5', '#F72585', '#4361EE', '#4CC9F0', '#7B2CBF'],
    'pld': ['#fb8500', '#ffb703', '#219ebc', '#023047', '#8ecae6'],
    'intercambio': ['#0057B8', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9', '#F0E442'],
    'geracao': ['#1B4332', '#2D6A4F', '#40916C', '#52B788', '#74C69D', '#95D5B2'],
    'renovaveis': ['#118AB2', '#06D6A0', '#FFD166', '#073B4C', '#EF476F'],
    'constrained': ['#FF6B35', '#F7C59F', '#D7263D', '#FF9F1C', '#C44536', '#FFD166'],
    'cvu': ['#FF6B35', '#F08A4B', '#C44536', '#8F2D56', '#F7B267', '#D1495B', '#7A284B'],
    'confiabilidade': ['#9D0208', '#D00000', '#DC2F02', '#E85D04', '#F48C06'],
}

THRESHOLDS = {
    'FREQ': {'min': 59.90, 'max': 60.10},
    'EAR': {'alerta_min': 30.0},
    'CMO': {'critico': 250.0}
}

print(f"✅ Célula 2: {len(datasets_selecionados)} datasets mapeados e estilos configurados.")


✅ Célula 2: 31 datasets mapeados e estilos configurados.


In [3]:
def _resource_timestamp(resource):
    return resource.get('last_modified') or resource.get('created') or ''


_slug_cache = {}
_data_cache = {}


def reset_ons_caches():
    """Reset metadata/data caches when the acquisition cell is re-executed."""
    _slug_cache.clear()
    _data_cache.clear()
    if 'datasets_selecionados' in globals():
        _slug_cache.update({slug: slug for slug in datasets_selecionados.values()})


reset_ons_caches()

CCEE_PLD_MEDIA_DIARIA_URL = 'https://pda-download.ccee.org.br/T09SGpnfRN-2ZaeWfHgrMw/content'
CCEE_PLD_MEDIA_DIARIA_PAGE = 'https://dadosabertos.ccee.org.br/dataset/pld_media_diaria'
CCEE_PLD_FALLBACK_B64 = '...'

def _normalize_ccee_pld(raw_bytes):
    df = pd.read_csv(io.BytesIO(raw_bytes), sep=';', encoding='iso-8859-2')
    df = df.rename(columns={'DIA':'dat_referencia','SUBMERCADO':'nom_subsistema','PLD_MEDIA_DIA':'val_pld'})
    df['dat_referencia'] = pd.to_datetime(df['dat_referencia'], format='%d/%m/%Y', errors='coerce', utc=True)
    df['val_pld'] = pd.to_numeric(df['val_pld'], errors='coerce')
    mapa = {'NORTE':'Norte','NORDESTE':'Nordeste','SUL':'Sul','SUDESTE':'Sudeste'}
    df['nom_subsistema'] = df['nom_subsistema'].astype(str).str.strip().str.upper().map(mapa).fillna(df['nom_subsistema'])
    return df.dropna(subset=['dat_referencia','val_pld']).sort_values('dat_referencia')

CCEE_PLD_STATUS_FILE = Path('.ccee_pld_status.json')
CCEE_PLD_RETRY_HOURS = 24


def _ccee_pld_fallback_result():
    df = _normalize_ccee_pld(base64.b64decode(CCEE_PLD_FALLBACK_B64))
    return df, CCEE_PLD_MEDIA_DIARIA_PAGE


def get_ccee_pld_media_diaria():
    """Obtém o PLD uma vez e usa contingência oficial durante o cooldown do 403."""
    agora = time.time()
    try:
        if CCEE_PLD_STATUS_FILE.exists():
            status = json.loads(CCEE_PLD_STATUS_FILE.read_text(encoding='utf-8'))
            bloqueado_ate = float(status.get('blocked_until', 0) or 0)
            if bloqueado_ate > agora:
                return _ccee_pld_fallback_result()
    except Exception:
        pass

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Accept': 'text/csv,application/octet-stream,*/*',
        'Referer': 'https://dadosabertos.ccee.org.br/'
    }
    try:
        resp = SESSION.get(CCEE_PLD_MEDIA_DIARIA_URL, headers=headers, timeout=(10, 30))
        resp.raise_for_status()
        return _normalize_ccee_pld(resp.content), CCEE_PLD_MEDIA_DIARIA_PAGE
    except Exception as exc:
        status_code = getattr(getattr(exc, 'response', None), 'status_code', None)
        if status_code == 403:
            mensagem = 'acesso automatizado bloqueado pela CCEE (403)'
        else:
            mensagem = str(exc)
        try:
            CCEE_PLD_STATUS_FILE.write_text(json.dumps({
                'blocked_until': agora + CCEE_PLD_RETRY_HOURS * 3600,
                'reason': mensagem,
                'recorded_at': agora
            }), encoding='utf-8')
        except Exception:
            pass
        print(f'   ⚠ PLD CCEE: {mensagem}; usando cópia oficial de contingência. Nova tentativa automática após {CCEE_PLD_RETRY_HOURS}h.', flush=True)
        return _ccee_pld_fallback_result()

def _find_date_column(columns):
    candidatos = ('din_instante', 'dat_referencia', 'momento', 'data', 'din_medicao', 'dat_medicao', 'dat_iniciosemana', 'dat_fimsemana')
    for col in columns:
        nome = str(col).lower()
        if any(c == nome or c in nome for c in candidatos):
            return col
    return None


def _filter_chunk(chunk, date_col, start_date, end_date):
    if not date_col or date_col not in chunk.columns:
        return chunk
    datas = pd.to_datetime(chunk[date_col], errors='coerce', utc=True)
    mask = datas.notna()
    if start_date is not None:
        mask &= datas >= start_date
    if end_date is not None:
        mask &= datas <= end_date
    resultado = chunk.loc[mask].copy()
    resultado[date_col] = datas.loc[mask]
    return resultado


def _read_tabular(download_url, fmt, periodo_dias=30, timeout=(5, 20), max_bytes=80_000_000):
    """Read ONS data with bounded memory and early period filtering where possible."""
    fmt = (fmt or '').upper()
    end_date = pd.Timestamp.now(tz='UTC') if periodo_dias is not None else None
    start_date = end_date - pd.Timedelta(days=periodo_dias) if periodo_dias is not None else None

    if fmt == 'PARQUET':
        # Try predicate pushdown for common ONS date columns; fall back safely if the
        # remote engine cannot apply a filter or the column has another name.
        for date_col in ('din_instante', 'dat_referencia', 'din_medicao', 'data', 'dat_iniciosemana', 'dat_fimsemana'):
            try:
                return pd.read_parquet(download_url, filters=[(date_col, '>=', start_date.to_datetime64())]) if start_date is not None else pd.read_parquet(download_url)
            except Exception:
                continue
        return pd.read_parquet(download_url)

    with SESSION.get(download_url, timeout=timeout, stream=True) as resp:
        resp.raise_for_status()
        content_length = int(resp.headers.get('Content-Length') or 0)
        if content_length and content_length > max_bytes:
            raise ValueError(f'recurso excede o limite de {max_bytes // 1_000_000} MB')
        with tempfile.NamedTemporaryFile(suffix='.data', delete=False) as temp:
            temp_path = Path(temp.name)
            total = 0
            try:
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    total += len(chunk)
                    if total > max_bytes:
                        raise ValueError(f'recurso excede o limite de {max_bytes // 1_000_000} MB')
                    temp.write(chunk)
            except Exception:
                temp_path.unlink(missing_ok=True)
                raise

    try:
        if fmt == 'XLSX':
            return pd.read_excel(temp_path)
        if fmt == 'JSON':
            data = json.loads(temp_path.read_text(encoding='utf-8-sig', errors='replace'))
            if isinstance(data, dict):
                for chave in ('records', 'result', 'data', 'results'):
                    if isinstance(data.get(chave), list):
                        return pd.json_normalize(data[chave])
                if isinstance(data.get('result'), dict) and isinstance(data['result'].get('records'), list):
                    return pd.json_normalize(data['result']['records'])
            return pd.json_normalize(data)

        chunks = []
        read_kwargs = dict(chunksize=100_000, low_memory=False, encoding='utf-8-sig')
        try:
            iterator = pd.read_csv(temp_path, sep=None, engine='python', **read_kwargs)
            for chunk in iterator:
                date_col = _find_date_column(chunk.columns)
                chunks.append(_filter_chunk(chunk, date_col, start_date, end_date))
        except Exception:
            chunks = []
            for sep in (';', ','):
                try:
                    iterator = pd.read_csv(temp_path, sep=sep, **read_kwargs)
                    for chunk in iterator:
                        date_col = _find_date_column(chunk.columns)
                        chunks.append(_filter_chunk(chunk, date_col, start_date, end_date))
                    if chunks:
                        break
                except Exception:
                    chunks = []
        if chunks:
            return pd.concat(chunks, ignore_index=True)
        return pd.read_csv(temp_path, sep=None, engine='python', encoding='latin-1')
    finally:
        temp_path.unlink(missing_ok=True)

def _resolve_slug(dataset_slug):
    """Tenta encontrar o slug real no catálogo CKAN caso o slug direto retorne 404."""
    try:
        search_url = f"https://dados.ons.org.br/api/3/action/package_search?q={dataset_slug}&rows=5"
        resp = SESSION.get(search_url, timeout=(5, 10))
        resp.raise_for_status()
        results = resp.json().get('result', {}).get('results', [])
        for pkg in results:
            if pkg.get('name') == dataset_slug:
                return dataset_slug
            if dataset_slug in pkg.get('name', ''):
                return pkg['name']
        return None
    except Exception:
        return None

def get_ons_resource(dataset_slug, periodo_dias=30):
    if dataset_slug in _data_cache:
        return _data_cache[dataset_slug]
    api_url = f"https://dados.ons.org.br/api/3/action/package_show?id={dataset_slug}"
    try:
        response = SESSION.get(api_url, timeout=(5, 15))
        if response.status_code != 200:
            slug_real = _resolve_slug(dataset_slug)
            if not slug_real or slug_real == dataset_slug:
                result = (None, None)
                _data_cache[dataset_slug] = result
                return result
            dataset_slug = slug_real
            response = SESSION.get(f"https://dados.ons.org.br/api/3/action/package_show?id={dataset_slug}", timeout=(5, 15))
        response.raise_for_status()
        resources = response.json().get('result', {}).get('resources', [])
        valid = [r for r in resources if r.get('url') and r.get('format', '').upper() in ('CSV', 'PARQUET', 'JSON', 'XLSX')]
        if not valid:
            result = (None, None)
        else:
            selected = max(valid, key=lambda r: (_resource_timestamp(r), r.get('name', '')))
            try:
                result = (_read_tabular(selected['url'], selected.get('format'), periodo_dias=periodo_dias), selected['url'])
            except Exception as exc:
                print(f"⚠️ Falha no recurso mais recente de {dataset_slug}: {exc}")
                result = (None, None)
        _data_cache[dataset_slug] = result
        return result
    except Exception as exc:
        print(f"⚠️ Erro na API ONS para {dataset_slug}: {exc}")
        _data_cache[dataset_slug] = (None, None)
        return _data_cache[dataset_slug]


print("✅ Célula 3: coleta em chunks, filtro antecipado, cache resetável e 8 workers prontos.")


✅ Célula 3: coleta em chunks, filtro antecipado, cache resetável e 8 workers prontos.


In [4]:
def get_fallback_data(nome_indicador):
    """Gera dados sintéticos caso a API do ONS falhe ou o recurso não exista."""
    datas = pd.date_range(end=pd.Timestamp.now(), periods=30, freq='D')
    if 'Armazenamento' in nome_indicador or 'EAR' in nome_indicador:
        valores = np.random.uniform(35, 65, size=30)
    elif 'CMO' in nome_indicador or 'PLD' in nome_indicador:
        valores = np.random.uniform(50, 200, size=30)
    else:
        valores = np.random.uniform(5000, 7000, size=30)

    return pd.DataFrame({'din_instante': datas, 'valor': valores})

In [5]:
# Diagnóstico removido da execução normal para não duplicar requests.
# Descomente e execute manualmente se precisar validar slugs:
#
# resultados_diag = []
# for nome, slug in datasets_selecionados.items():
#     api_url = f"https://dados.ons.org.br/api/3/action/package_show?id={slug}"
#     try:
#         resp = SESSION.get(api_url, timeout=15)
#         status = "OK" if resp.status_code == 200 else f"HTTP {resp.status_code}"
#     except Exception as e:
#         status = str(e)
#     resultados_diag.append((nome, slug, status))
# import pandas as pd
# pd.DataFrame(resultados_diag, columns=["Indicador","Slug","Status"])

print("✅ Célula 4 (diagnóstico): desativada — execute manualmente se necessário.")


✅ Célula 4 (diagnóstico): desativada — execute manualmente se necessário.


In [6]:
def _prepare_dataframe(df):
    if df is None or df.empty:
        return df
    trabalho = df.copy()
    for col in trabalho.columns:
        if pd.api.types.is_object_dtype(trabalho[col]) or pd.api.types.is_string_dtype(trabalho[col]):
            convertido = pd.to_numeric(trabalho[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')
            if convertido.notna().mean() >= 0.8:
                trabalho[col] = convertido
    return trabalho


def _tcol(df):
    if df is None or df.empty: return 'din_instante'
    candidatos = ('din_instante', 'dat_referencia', 'momento', 'data', 'din_medicao', 'dat_medicao', 'dat_iniciosemana', 'dat_fimsemana')
    for col in df.columns:
        nome = str(col).lower()
        if any(c == nome or c in nome for c in candidatos): return col
    return df.columns[0]


def _vcol(df, candidatos, tcol=None):
    if df is None or df.empty:
        return None
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if tcol and tcol in num_cols:
        num_cols.remove(tcol)
    if not num_cols:
        return None
    for cand in candidatos:
        for col in num_cols:
            if cand.lower() in col.lower():
                return col
    return next((c for c in num_cols if c.lower().startswith(('val_', 'vl_'))), num_cols[0])


def _filter_period(df, tcol, period_str='30d'):
    if df is None or df.empty or tcol not in df.columns: return df
    trabalho = df.copy()
    trabalho[tcol] = pd.to_datetime(trabalho[tcol], errors='coerce', utc=True)
    trabalho = trabalho.dropna(subset=[tcol]).sort_values(tcol, kind='stable')
    if trabalho.empty: return trabalho
    days = PERIOD_DAYS_MAP.get(period_str, 30)
    return trabalho[trabalho[tcol] >= trabalho[tcol].max() - timedelta(days=days)].copy()


def _sorted_ready(df, tcol):
    if df is None or df.empty or tcol not in df.columns: return df
    if not pd.api.types.is_datetime64_any_dtype(df[tcol]):
        df = df.copy(); df[tcol] = pd.to_datetime(df[tcol], errors='coerce', utc=True)
    return df.sort_values(tcol, kind='stable') if not df[tcol].is_monotonic_increasing else df


def _get_kpi_recente(df, tcol, vcol):
    if df is None or df.empty or vcol is None or vcol not in df.columns: return None
    validos = df.loc[df[vcol].notna(), vcol]
    return validos.iloc[-1] if not validos.empty else None


def _calc_variacao(df, tcol, vcol):
    if df is None or len(df) < 2 or vcol is None or vcol not in df.columns: return None
    trabalho = df.dropna(subset=[tcol, vcol])
    if len(trabalho) < 2: return None
    atual = trabalho.iloc[-1]
    anteriores = trabalho[trabalho[tcol] <= atual[tcol] - pd.Timedelta(days=1)]
    anterior = anteriores.iloc[-1] if not anteriores.empty else trabalho.iloc[-2]
    if anterior[vcol] == 0: return None
    return ((atual[vcol] - anterior[vcol]) / abs(anterior[vcol])) * 100


def _status_color(val, kpi_type='DEFAULT'):
    if val is None: return '#888888', 'Sem dados'
    if kpi_type == 'EAR': return ('#E63946', 'Baixo') if val < THRESHOLDS['EAR']['alerta_min'] else ('#00B050', 'Normal')
    if kpi_type == 'CMO': return ('#E63946', 'Elevado') if val > THRESHOLDS['CMO']['critico'] else ('#00B050', 'Normal')
    return '#0078D4', 'OK'


def _checar_frescor(df, tcol, dias_alerta=5):
    if df is None or df.empty or tcol not in df.columns: return None
    max_date = pd.to_datetime(df[tcol], errors='coerce', utc=True).max()
    if pd.isna(max_date): return None
    defasagem = (datetime.now(timezone.utc) - max_date).days
    return defasagem if defasagem > dias_alerta else None


def _format_kpi(value, unit):
    if value is None or pd.isna(value): return 'N/A'
    if unit == '%':
        return f'{value:.1f}%' if abs(value) <= 1000 else f'{value:,.1f}'
    if abs(value) >= 1e6: return f'{value / 1e6:.2f}M'
    if abs(value) >= 1e3: return f'{value / 1e3:.1f}k'
    return f'{value:.2f}'

def _downsample_for_chart(df, tcol, vcol, max_points=500, max_series=24):
    """Reduce only chart payload size while preserving the full export dataframe."""
    if df is None or df.empty or tcol not in df.columns or vcol not in df.columns:
        return df
    group_col = next((c for c in ('nom_subsistema', 'nom_ree', 'nom_bacia', 'nom_usina') if c in df.columns), None)
    def sample_frame(frame):
        frame = frame.sort_values(tcol, kind='stable')
        if len(frame) <= max_points:
            return frame
        indices = pd.Index(np.linspace(0, len(frame) - 1, max_points, dtype=int)).unique()
        return frame.iloc[indices]
    if group_col and df[group_col].nunique(dropna=False) > 1:
        counts = df[group_col].value_counts(dropna=False).head(max_series)
        partes = [sample_frame(df[df[group_col].eq(g) if pd.notna(g) else df[group_col].isna()]) for g in counts.index]
        return pd.concat(partes, ignore_index=True) if partes else df.head(0)
    return sample_frame(df)


print("✅ Célula 4: frames ordenados uma vez, coluna temporal excluída e KPIs robustos.")


✅ Célula 4: frames ordenados uma vez, coluna temporal excluída e KPIs robustos.


In [7]:
def hex_to_rgba(hex_str, alpha=0.15):
    hex_str = str(hex_str).lstrip('#')
    try:
        r, g, b = tuple(int(hex_str[i:i + 2], 16) for i in (0, 2, 4))
        return f'rgba({r}, {g}, {b}, {alpha})'
    except Exception:
        return f'rgba(0, 120, 212, {alpha})'

def _json_safe(value):
    if isinstance(value, np.ndarray):
        return _json_safe(value.tolist())
    if isinstance(value, dict):
        return {k: _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    if value is None or value is pd.NaT:
        return None
    if isinstance(value, (float, np.floating)) and not np.isfinite(value):
        return None
    if isinstance(value, (np.datetime64,)) and np.isnat(value):
        return None
    if isinstance(value, pd.Timestamp) and pd.isna(value):
        return None
    try:
        missing = pd.isna(value)
        if isinstance(missing, (bool, np.bool_)) and missing:
            return None
    except Exception:
        pass
    return value


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.ndarray, np.generic)):
            return obj.tolist() if isinstance(obj, np.ndarray) else obj.item()
        if isinstance(obj, (datetime, pd.Timestamp)):
            return obj.isoformat()
        return super(NumpyEncoder, self).default(obj)

def render_intercambio_chart(df, label, tc, vc, unit='MWmed', region_filter=None):
    """Gráfico direcional: recebimento acima de zero e envio abaixo de zero."""
    if df is None or df.empty:
        fig = go.Figure()
        fig.add_annotation(text='Dados indisponíveis', showarrow=False)
        return fig
    origem_col = next((c for c in ('nom_subsistema_origem', 'subsistema_origem') if c in df.columns), None)
    destino_col = next((c for c in ('nom_subsistema_destino', 'subsistema_destino') if c in df.columns), None)
    if not origem_col or not destino_col or tc not in df.columns or vc not in df.columns:
        fig = go.Figure()
        fig.add_annotation(text='Colunas de intercâmbio indisponíveis', showarrow=False)
        return fig
    trabalho = df[[tc, origem_col, destino_col, vc]].copy()
    trabalho['_data'] = pd.to_datetime(trabalho[tc], errors='coerce').dt.floor('D')
    trabalho['_origem'] = trabalho[origem_col].astype(str).str.upper().str.strip()
    trabalho['_destino'] = trabalho[destino_col].astype(str).str.upper().str.strip()
    trabalho['_valor'] = pd.to_numeric(trabalho[vc], errors='coerce')
    trabalho = trabalho.dropna(subset=['_data', '_valor'])
    trabalho = trabalho[~trabalho['_origem'].isin(['', 'NAN']) & ~trabalho['_destino'].isin(['', 'NAN'])]
    if region_filter:
        region_filter = str(region_filter).upper().strip()
        trabalho = trabalho[((trabalho['_origem'] == 'SUDESTE') & (trabalho['_destino'] == region_filter)) | ((trabalho['_origem'] == region_filter) & (trabalho['_destino'] == 'SUDESTE'))]
    regioes = [r for r in ('SUL', 'NORDESTE') if (
        ((trabalho['_origem'] == 'SUDESTE') & (trabalho['_destino'] == r)).any()
        or ((trabalho['_origem'] == r) & (trabalho['_destino'] == 'SUDESTE')).any()
    )]
    if region_filter:
        regioes = [region_filter] if not trabalho.empty else []
    elif not regioes:
        regioes = [r for r in sorted(set(trabalho['_origem']) | set(trabalho['_destino'])) if r != 'SUDESTE']
    if not regioes:
        fig = go.Figure()
        fig.add_annotation(text='Não há intercâmbios no período', showarrow=False)
        return fig
    fig = make_subplots(rows=len(regioes), cols=1, shared_xaxes=True, vertical_spacing=0.10)
    cores = {'recebendo': '#4472C4', 'enviado': '#ED7D31'}
    nomes = {'SUL': 'Sul', 'NORDESTE': 'Nordeste', 'NORTE': 'Norte'}
    for row, regiao in enumerate(regioes, start=1):
        recebido = trabalho[(trabalho['_origem'] == 'SUDESTE') & (trabalho['_destino'] == regiao)].groupby('_data', as_index=False)['_valor'].sum()
        enviado = trabalho[(trabalho['_origem'] == regiao) & (trabalho['_destino'] == 'SUDESTE')].groupby('_data', as_index=False)['_valor'].sum()
        nome = nomes.get(regiao, regiao.title())
        fig.add_trace(go.Bar(x=recebido['_data'], y=recebido['_valor'], name=f'{nome} recebendo do Sudeste', marker_color=cores['recebendo'], legendgroup='recebendo', showlegend=True, hovertemplate=f'{nome} recebendo do Sudeste<br>%{{x|%d/%m/%Y}}<br>%{{y:,.0f}} {unit}<extra></extra>'), row=row, col=1)
        fig.add_trace(go.Bar(x=enviado['_data'], y=-enviado['_valor'].abs(), name=f'{nome} enviado para Sudeste', marker_color=cores['enviado'], legendgroup='enviado', showlegend=True, hovertemplate=f'{nome} enviado para Sudeste<br>%{{x|%d/%m/%Y}}<br>%{{y:,.0f}} {unit}<extra></extra>'), row=row, col=1)
        fig.update_yaxes(title_text=unit or 'MW', zeroline=True, zerolinecolor='#777', zerolinewidth=1, gridcolor='#d9d9d9', row=row, col=1)
        fig.update_xaxes(showgrid=False, tickformat='%b-%y', dtick='M1', row=row, col=1)
    fig.update_layout(template='plotly_white', height=max(430, 220 * len(regioes)), margin=dict(l=58, r=145, t=16, b=48), barmode='relative', bargap=0.04, hovermode='x unified', showlegend=True, legend=dict(orientation='v', x=1.01, xanchor='left', y=1, font=dict(size=10)), font=dict(color='#2d2d3a'))
    return fig

def render_chart(df, label, tc, vc, unit='', category='default'):
    fig = go.Figure()
    if df is None or df.empty or tc not in df.columns or vc not in df.columns:
        fig.add_annotation(text="Dados indisponíveis", showarrow=False)
        return fig

    palette = PALETAS.get(category, PALETAS['default'])
    group_col = next((c for c in ('nom_subsistema', 'nom_ree', 'nom_bacia', 'nom_usina') if c in df.columns), None)

    if group_col and df[group_col].nunique() > 1:
        for i, (name, g_df) in enumerate(df.groupby(group_col)):
            fig.add_trace(go.Scatter(x=g_df[tc], y=g_df[vc], name=str(name), line=dict(color=palette[i % len(palette)], width=2),
                                         hovertemplate=f'<b>{str(name)}</b><br>Data: %{{x|%d/%m/%Y}}<br>Valor: %{{y:,.2f}} {unit}<extra></extra>'))
    else:
        fig.add_trace(go.Scatter(x=df[tc], y=df[vc], name=label, fill='tozeroy', line=dict(color=palette[0], width=2),
                             hovertemplate=f'<b>{label}</b><br>Data: %{{x|%d/%m/%Y}}<br>Valor: %{{y:,.2f}} {unit}<extra></extra>'))

    fig.update_layout(template='plotly_white', height=250, margin=dict(l=10, r=10, t=10, b=10), showlegend=True,
                      yaxis=dict(title=unit or None),
                      hovermode='x unified', hoverlabel=dict(bgcolor='#ffffff', font=dict(color='#1f2937', size=12)))
    return fig

def render_cvu_combo_chart(df, label, tc, vc, unit='R$/MWh'):
    """Renderiza o CVU consolidado como colunas por usina na última data disponível.

    A preparação dos dados, os filtros e os KPIs continuam sendo executados fora
    desta função; aqui é alterado somente o formato visual do gráfico do card.
    """
    fig = go.Figure()
    if df is None or df.empty or tc not in df.columns or vc not in df.columns:
        fig.add_annotation(text='Dados indisponíveis', showarrow=False)
        return fig

    cols = [tc, vc]
    if 'nom_subsistema' in df.columns:
        cols.append('nom_subsistema')
    if 'nom_usina' in df.columns:
        cols.append('nom_usina')
    trabalho = df[cols].copy().dropna(subset=[tc, vc]).sort_values(tc)
    if trabalho.empty:
        fig.add_annotation(text='Dados indisponíveis', showarrow=False)
        return fig

    # Uma coluna por usina, usando o valor mais recente do recorte carregado.
    # A ordenação crescente reproduz o layout de empilhamento térmico da referência.
    if 'nom_usina' in trabalho.columns:
        trabalho['_cvu_usina'] = trabalho['nom_usina'].astype(str).str.strip()
        trabalho = trabalho[trabalho['_cvu_usina'].ne('')].copy()
        trabalho = trabalho.drop_duplicates(subset=['_cvu_usina'], keep='last')
    else:
        trabalho['_cvu_usina'] = label
        trabalho = trabalho.tail(1)

    trabalho[vc] = pd.to_numeric(trabalho[vc], errors='coerce')
    trabalho = trabalho.dropna(subset=[vc]).sort_values(vc, kind='stable')
    if trabalho.empty:
        fig.add_annotation(text='Dados indisponíveis', showarrow=False)
        return fig

    cores_regiao = {'Norte':'#0072B2','Sul':'#009E73','Sudeste/Centro-Oeste':'#D55E00','Nordeste':'#CC79A7'}
    if 'nom_subsistema' in trabalho.columns:
        cores = [cores_regiao.get(str(regiao), '#A96E5F') for regiao in trabalho['nom_subsistema']]
        regioes = trabalho['nom_subsistema'].astype(str).tolist()
    else:
        cores = ['#A96E5F'] * len(trabalho)
        regioes = [label] * len(trabalho)

    nomes = trabalho['_cvu_usina'].tolist()
    valores = trabalho[vc].tolist()
    datas = trabalho[tc].tolist()

    def _formatar_data_cvu(valor):
        try:
            convertido = pd.to_datetime(valor, errors='coerce')
            return convertido.strftime('%d/%m/%Y') if not pd.isna(convertido) else str(valor)[:10]
        except Exception:
            return str(valor)[:10]

    def _formatar_numero_cvu(valor):
        try:
            return f'{float(valor):,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
        except Exception:
            return str(valor)

    datas_formatadas = [_formatar_data_cvu(data) for data in datas]
    valores_formatados = [_formatar_numero_cvu(valor) for valor in valores]
    fig.add_trace(go.Bar(
        x=nomes,
        y=valores,
        name='CVU',
        marker=dict(color=cores, line=dict(color='rgba(100,60,50,0.30)', width=0.4)),
        hovertemplate=(
            '<b>%{x}</b><br>Região: %{customdata[0]}<br>'
            'Data: %{customdata[1]}<br>'
            f'CVU: %{{customdata[2]}} {unit}<extra></extra>'
        ),
        customdata=[[regiao, data, valor] for regiao, data, valor in zip(regioes, datas_formatadas, valores_formatados)],
    ))
    fig.update_layout(
        template='plotly_white',
        height=390,
        margin=dict(l=58, r=18, t=16, b=112),
        showlegend=False,
        hovermode='closest',
        hoverlabel=dict(bgcolor='#ffffff', font=dict(color='#1f2937', size=12)),
        bargap=0.16,
        xaxis=dict(
            title='',
            showgrid=False,
            showline=True,
            linecolor='#d9d9d9',
            tickangle=-90,
            tickfont=dict(size=8, color='#6b7280'),
            automargin=True,
        ),
        yaxis=dict(
            title=unit,
            showgrid=True,
            gridcolor='#dedede',
            zeroline=False,
            rangemode='tozero',
            tickfont=dict(size=10, color='#6b7280'),
            automargin=True,
        ),
    )
    return fig

def _df_to_csv_b64(df):
    import base64
    if df is None or df.empty:
        return None
    try:
        csv_str = df.to_csv(index=False, encoding='utf-8-sig')
        return base64.b64encode(csv_str.encode('utf-8-sig')).decode('ascii')
    except Exception:
        return None


def _df_to_xlsx_b64(df):
    # Converte o dataframe do card em um arquivo XLSX embutido no download.
    import base64
    if df is None or df.empty:
        return None
    try:
        export_df = df.copy()
        for col in export_df.columns:
            try:
                if pd.api.types.is_datetime64tz_dtype(export_df[col]):
                    export_df[col] = export_df[col].dt.tz_localize(None)
            except Exception:
                pass
        buffer = io.BytesIO()
        with pd.ExcelWriter(buffer, engine='openpyxl') as writer:
            export_df.to_excel(writer, index=False, sheet_name='Dados')
        return base64.b64encode(buffer.getvalue()).decode('ascii')
    except Exception:
        return None


def card_wrap(fig, title, url, status, description="", kpi_val=None, kpi_unit=None,
              kpi_delta=None, status_color='#0078D4', status_label='', tab_cat='transmissao',
              is_fallback=False, defasagem_dias=None, df_export=None,
              dataset_slug='', vcol='', accent_color='', chart_mode='', chart_unit=''):
    try:
        figure_spec = json.dumps(_json_safe(fig.to_dict()), cls=NumpyEncoder, ensure_ascii=False, allow_nan=False)
        html_fig = f'<div class="plotly-lazy" data-spec="{html_lib.escape(figure_spec)}" style="height:{390 if chart_mode == "cvu-consolidado" else 260}px;"></div>'
    except Exception as e:
        html_fig = f'<div>Erro: {str(e)}</div>'

    csv_b64 = ''
    xlsx_b64 = ''
    download_links = []
    if df_export is not None and not df_export.empty:
        csv_b64 = _df_to_csv_b64(df_export) or ''
        xlsx_b64 = _df_to_xlsx_b64(df_export) or ''
        nome_arquivo = str(title).replace('—', '-')
        nome_arquivo = unicodedata.normalize('NFKD', nome_arquivo).encode('ascii', 'ignore').decode('ascii')
        nome_arquivo = nome_arquivo.replace(' ', '_').replace('/', '-')
        if xlsx_b64:
            download_links.append(
                f'<a class="download-link download-excel" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{xlsx_b64}" '
                f'download="{nome_arquivo}.xlsx" title="Baixar dados em Excel">Excel</a>'
            )
        if csv_b64:
            download_links.append(
                f'<a class="download-link download-csv" href="data:text/csv;charset=utf-8;base64,{csv_b64}" '
                f'download="{nome_arquivo}.csv" title="Baixar dados em CSV">CSV</a>'
            )

    status_class = 'success' if status == '✓' else 'warning'
    source_link = ''
    if url or dataset_slug:
        # Abre a página do conjunto no catálogo, não o arquivo bruto.
        # O arquivo bruto continua disponível nos links CSV/Excel separados.
        is_ons_dataset = bool(dataset_slug) and not str(url).lower().startswith(('https://dadosabertos.ccee', 'http://dadosabertos.ccee'))
        catalog_url = f'https://dados.ons.org.br/dataset/{dataset_slug}' if is_ons_dataset else str(url or '')
        source_link = f'<a class="source-link" href="{html_lib.escape(catalog_url, quote=True)}" target="_blank" rel="noopener">Fonte oficial ↗</a>' if catalog_url else ''

    usina_select = ''
    if chart_mode in ('cvu-combo', 'cvu-consolidado') and df_export is not None and 'nom_usina' in df_export.columns:
        usinas = sorted({str(v).strip() for v in df_export['nom_usina'].dropna() if str(v).strip()})
        options = '<option value="__TODAS__">Todas as usinas</option>' + ''.join(
            f'<option value="{html_lib.escape(u, quote=True)}">{html_lib.escape(u)}</option>' for u in usinas
        )
        usina_select = (f'<label class="control-field">Usina'
                        f'<select class="usina-select">{options}</select></label>')

    accent_style = f' style="--card-accent:{html_lib.escape(accent_color, quote=True)};"' if accent_color else ''
    region_key = title.lower().replace(' ', '-').replace('/', '-')
    source_data_url = str(url) if str(url).lower().split('?')[0].endswith('.csv') else ''
    card_kind = 'pld-card' if chart_mode == 'pld-regiao' else ('cvu-consolidado' if chart_mode == 'cvu-consolidado' else '')
    delta_html = ''
    if kpi_delta is not None:
        delta_color = '#16845b' if kpi_delta >= 0 else '#c0392b'
        delta_prefix = '+' if kpi_delta >= 0 else ''
        delta_html = f'<span class="card-delta" style="color:{delta_color};">{delta_prefix}{kpi_delta:.1f}% vs. período anterior</span>'
    fallback_html = '<span class="data-badge fallback">Contingência</span>' if is_fallback else ''
    freshness_html = f'<span class="freshness">{defasagem_dias}d de defasagem</span>' if defasagem_dias is not None else '<span class="freshness">Atualização verificada</span>'

    return f'''<article class="card-item cat-{tab_cat} {region_key} {card_kind}"{accent_style} data-region="{html_lib.escape(title, quote=True)}" data-slug="{html_lib.escape(str(dataset_slug), quote=True)}" data-vcol="{html_lib.escape(str(vcol), quote=True)}" data-chart-mode="{html_lib.escape(str(chart_mode), quote=True)}" data-unit="{html_lib.escape(str(chart_unit), quote=True)}" data-source-url="{html_lib.escape(source_data_url, quote=True)}" data-csv-b64="{csv_b64}" data-xlsx-b64="{xlsx_b64}">
        <div class="card-header">
            <div class="card-title-wrap">
                <span class="status-dot {status_class}"></span>
                <strong class="card-title" onclick="abrirDetalhe(this)" title="Abrir detalhamento">{html_lib.escape(title)}</strong>
                {fallback_html}
            </div>
            <div class="card-actions">{source_link}{''.join(download_links)}</div>
        </div>
        <div class="card-kpi-value">{kpi_val if kpi_val is not None else '—'} <span class="card-kpi-unit">{kpi_unit or ''}</span></div>
        <div class="card-meta">{delta_html}{freshness_html}{f'<span class="status-label">{html_lib.escape(status_label)}</span>' if status_label else ''}</div>
        <div class="card-controls">
            <label class="control-field">De<input type="date" class="date-inicio"></label>
            <label class="control-field">Até<input type="date" class="date-fim"></label>
            <button class="btn-atualizar" type="button">Atualizar</button>
            {usina_select}
            <span class="msg-status" role="status"></span>
        </div>
        {description if (description and chart_mode == 'cvu-consolidado') else ''}
        {html_fig}
    </article>'''



In [8]:
def _get_unit(slug, nome):
    if 'CVU' in nome.upper() or 'USINA TERMICA' in nome.upper() or 'USINA TÉRMICA' in nome.upper(): return 'R$/MWh'
    if any(x in nome.upper() for x in ['CARGA', 'GERACAO', 'ENA', 'INTERCAMBIO']): return 'MWmed'
    if 'EAR' in nome.upper() or 'ARMAZENAMENTO' in nome.upper(): return '%'
    if 'CMO' in nome.upper() or 'PLD' in nome.upper(): return 'R$/MWh'
    return ''

def _get_category(slug, nome):
    n = nome.upper()
    if 'CVU' in n or 'USINA TERMICA' in n or 'USINA TÉRMICA' in n: return 'cvu'
    if 'EAR' in n or 'ARMAZENAMENTO' in n: return 'armazenamento'
    if 'ENA' in n: return 'ena'
    if 'CARGA' in n: return 'carga'
    if 'CMO' in n or 'PLD' in n: return 'cmo'
    if 'GERACAO' in n or 'EOLICA' in n or 'SOLAR' in n: return 'geracao'
    return 'default'

def _get_tab_category(slug, nome):
    n = nome.upper()
    if 'CONSTRAINED' in n or 'COFF' in n:
        return 'constrained'
    if 'CVU' in n or 'USINA TERMICA' in n or 'USINA TÉRMICA' in n:
        return 'mercado'
    cat = _get_category(slug, nome)
    mapping = {'armazenamento': 'hidrologia', 'ena': 'hidrologia', 'carga': 'carga', 'cmo': 'mercado', 'geracao': 'geracao'}
    return mapping.get(cat, 'transmissao')

print('✅ Funções de mapeamento de metadados adicionadas.')

✅ Funções de mapeamento de metadados adicionadas.


In [9]:
dados_coletados = {}
cards_html = []
fallback_indicadores = []
falhas_indicadores = []
alertas_count = 0
kpi_carga_val = "N/A"
kpi_ear_val = "N/A"
total_indicadores = len(datasets_selecionados)

periodo_dias = PERIOD_DAYS_MAP.get(PERIODO_ATUAL, 30)

# ── Fetch com print por slug para identificar qual trava ──────────────────────
def fetch_all_datasets(datasets_dict, periodo_dias, max_workers=6, timeout_por_slug=25):
    unique_slugs = list(dict.fromkeys(datasets_dict.values()))

    def _fetch_slug(slug):
        t = time.time()
        print(f"   ⬇ {slug}", flush=True)
        try:
            if slug == 'ccee-pld-media-diaria':
                result = slug, *get_ccee_pld_media_diaria()
            else:
                result = slug, *get_ons_resource(slug, periodo_dias=None)
            print(f"   ✓ {slug} ({time.time()-t:.1f}s)", flush=True)
            return result
        except Exception as exc:
            print(f"   ✗ {slug} ({time.time()-t:.1f}s) → {exc}", flush=True)
            return slug, None, None

    fetched = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_fetch_slug, slug): slug for slug in unique_slugs}
        for future in concurrent.futures.as_completed(futures):
            slug = futures[future]
            try:
                resolved_slug, df, url = future.result(timeout=timeout_por_slug)
                fetched[resolved_slug] = (df, url)
            except Exception:
                print(f"   ⏰ ABANDONADO: {slug}", flush=True)
                fetched[slug] = (None, None)

    # slugs que nunca voltaram viram fallback
    for slug in unique_slugs:
        if slug not in fetched:
            print(f"   ⏰ NÃO RETORNOU: {slug}", flush=True)
            fetched[slug] = (None, None)

    return {
        nome: {'slug': slug, 'df': fetched.get(slug, (None, None))[0], 'url': fetched.get(slug, (None, None))[1]}
        for nome, slug in datasets_dict.items()
    }

print(f"🚀 Iniciando coleta — {len(datasets_selecionados)} indicadores, {periodo_dias} dias\n")
t0 = time.time()

resultados_brutos = fetch_all_datasets(datasets_selecionados, periodo_dias, max_workers=6)

print(f"\n⏱ Fetch concluído em {time.time() - t0:.1f}s\n")

# ── Loop de processamento ─────────────────────────────────────────────────────
for nome_indicador, slug in datasets_selecionados.items():
    try:
        bruto = resultados_brutos.get(nome_indicador, {})
        df, url = bruto.get('df'), bruto.get('url')
        is_fallback = False

        if df is None or df.empty:
            # Para PLD, sempre tenta recuperar a contingência oficial regional antes
            # de recorrer ao fallback genérico de uma única série.
            if nome_indicador == 'PLD CCEE':
                try:
                    df, url = get_ccee_pld_media_diaria()
                except Exception:
                    df, url = None, None
            if df is None or df.empty:
                df = get_fallback_data(nome_indicador)
                url = f"https://dados.ons.org.br/dataset/{slug}"
                is_fallback = True
                status = '⚠'
                fallback_indicadores.append(nome_indicador)
            else:
                is_fallback = True
                status = '⚠'
        else:
            status = '✓'

        df = _prepare_dataframe(df)
        tc = _tcol(df)
        vc = _vcol(df, ['val_', 'valor', 'geracao', 'carga', 'ear', 'ena', 'cmo', 'pld'], tcol=tc)

        df_p = _filter_period(df, tc, PERIODO_ATUAL)
        df_p = _sorted_ready(df_p, tc)
        dados_coletados[nome_indicador] = df_p

        defasagem = None if is_fallback else _checar_frescor(df, tc)
        kpi_val = _get_kpi_recente(df_p, tc, vc)
        kpi_delta = _calc_variacao(df_p, tc, vc)

        unit = _get_unit(slug, nome_indicador)
        cat = _get_category(slug, nome_indicador)
        tab_cat = _get_tab_category(slug, nome_indicador)

        status_color, status_label = _status_color(kpi_val, 'EAR' if cat == 'armazenamento' else 'DEFAULT')

        if status_label in ('Baixo', 'Elevado') or status == '⚠':
            alertas_count += 1
        if nome_indicador == 'Carga' and kpi_val is not None:
            kpi_carga_val = _format_kpi(kpi_val, 'MW')
        if nome_indicador == 'Armazenamento' and kpi_val is not None:
            kpi_ear_val = _format_kpi(kpi_val, '%')

        if nome_indicador == 'PLD CCEE' and 'nom_subsistema' in df_p.columns:
            cores_pld = {'Norte':'#0072B2','Nordeste':'#D55E00','Sul':'#009E73','Sudeste':'#CC79A7'}
            ultima_data_pld = df_p[tc].max()
            ordem_pld = (df_p[df_p[tc].eq(ultima_data_pld)].groupby('nom_subsistema')[vc].mean().sort_values().index.tolist())
            for regiao in ordem_pld:
                df_regiao = df_p[df_p['nom_subsistema'].eq(regiao)].copy()
                if df_regiao.empty: continue
                kpi_regiao = _get_kpi_recente(df_regiao, tc, vc)
                delta_regiao = _calc_variacao(df_regiao, tc, vc)
                fig_regiao = render_chart(df_regiao, f'PLD {regiao}', tc, vc, unit=unit, category='pld')
                cards_html.append(card_wrap(
                    fig=fig_regiao, title=f'PLD {regiao}', url=url, status=status,
                    kpi_val=_format_kpi(kpi_regiao, unit), kpi_unit=unit, kpi_delta=delta_regiao, chart_unit=unit,
                    status_color=status_color, status_label=status_label, tab_cat='pld-regiao',
                    is_fallback=False, defasagem_dias=defasagem, df_export=df_regiao,
                    dataset_slug=slug, vcol=vc or '', accent_color=cores_pld.get(regiao, '#0072B2'), chart_mode='pld-regiao'))
                print(f" {status} PLD {regiao:<27} [PLD-CCEE] [OK]")
        elif nome_indicador == 'Intercâmbios entre Subsistemas' and not is_fallback:
            # Dois cards independentes, um por região, preservando as duas direções.
            origem_col_inter = next((c for c in ('nom_subsistema_origem', 'subsistema_origem') if c in df_p.columns), None)
            destino_col_inter = next((c for c in ('nom_subsistema_destino', 'subsistema_destino') if c in df_p.columns), None)
            for regiao_inter, nome_regiao_inter in (('SUL', 'Sul'), ('NORDESTE', 'Nordeste')):
                if not origem_col_inter or not destino_col_inter:
                    continue
                origem_inter = df_p[origem_col_inter].astype(str).str.upper().str.strip()
                destino_inter = df_p[destino_col_inter].astype(str).str.upper().str.strip()
                mascara_inter = ((origem_inter == 'SUDESTE') & (destino_inter == regiao_inter)) | ((origem_inter == regiao_inter) & (destino_inter == 'SUDESTE'))
                df_regiao_inter = df_p.loc[mascara_inter].copy()
                if df_regiao_inter.empty:
                    continue
                kpi_inter = _get_kpi_recente(df_regiao_inter, tc, vc)
                delta_inter = _calc_variacao(df_regiao_inter, tc, vc)
                fig_intercambio = render_intercambio_chart(df_regiao_inter, f'Intercâmbios — {nome_regiao_inter}', tc, vc, unit='MWmed', region_filter=regiao_inter)
                cards_html.append(card_wrap(
                    fig=fig_intercambio, title=f'Intercâmbios entre Subsistemas — {nome_regiao_inter}', url=url, status=status,
                    kpi_val=_format_kpi(kpi_inter, unit), kpi_unit=unit, kpi_delta=delta_inter, chart_unit=unit,
                    status_color=status_color, status_label=status_label, tab_cat=tab_cat,
                    is_fallback=False, defasagem_dias=defasagem, df_export=df,
                    dataset_slug=slug, vcol=vc or '', accent_color='#4472C4',
                    chart_mode='intercambio-direcional'))
                print(f" {status} Intercâmbios {nome_regiao_inter:<18} [{tab_cat.upper()}] [INTERCAMBIO-DIRECIONAL] [OK]")
        elif nome_indicador == 'CVU Usina Térmica' and not is_fallback and 'nom_subsistema' in df_p.columns:
            # Um único card detalhado, com todas as regiões e usinas.
            cores_regiao = {
                'Norte': '#0072B2',
                'Sul': '#009E73',
                'Sudeste/Centro-Oeste': '#D55E00',
                'Nordeste': '#CC79A7',
            }
            ultima_data_cvu = df_p[tc].max()
            recorte_ordenacao = df_p[df_p[tc].eq(ultima_data_cvu)]
            ordem_regioes = recorte_ordenacao.groupby('nom_subsistema')[vc].mean().sort_values().index.tolist()
            resumo = []
            for regiao in ordem_regioes:
                df_regiao = df_p[df_p['nom_subsistema'].eq(regiao)].copy()
                if df_regiao.empty: continue
                kpi_regiao = _get_kpi_recente(df_regiao, tc, vc)
                delta_regiao = _calc_variacao(df_regiao, tc, vc)
                cor = cores_regiao.get(regiao, '#0072B2')
                delta_txt = ('+' if delta_regiao is not None and delta_regiao >= 0 else '') + (f'{delta_regiao:.1f}%' if delta_regiao is not None else '—')
                resumo.append(f'<div class="cvu-region-summary" style="border-top:3px solid {cor};"><strong style="color:{cor};">{html_lib.escape(str(regiao))}</strong><span>{_format_kpi(kpi_regiao, unit)} {unit}</span><small>Variação: {delta_txt}</small></div>')
            resumo_html = '<div class="cvu-region-summary-grid">' + ''.join(resumo) + '</div><div class="cvu-help">Visão consolidada por região e usina. Use o seletor para isolar uma usina; as cores identificam os submercados.</div>'
            fig_cvu = render_cvu_combo_chart(df_p, 'CVU Usina Térmica', tc, vc, unit=unit)
            cards_html.append(card_wrap(
                fig=fig_cvu, title='CVU Usina Térmica — Visão Consolidada', url=url, status=status,
                kpi_val=f'{len(ordem_regioes)} regiões', kpi_unit='• todas as usinas', kpi_delta=None, chart_unit=unit,
                status_color=status_color, status_label=status_label, tab_cat='cvu-regiao',
                is_fallback=False, defasagem_dias=defasagem, description=resumo_html,
                df_export=df, dataset_slug=slug, vcol=vc or '', accent_color='#0072B2',
                chart_mode='cvu-consolidado',
            ))
            print(f" {status} CVU Usina Térmica — Visão Consolidada [CVU-CONSOLIDADO] [OK]")
        else:
            df_chart = _downsample_for_chart(df_p, tc, vc)
            fig = render_chart(df_chart, nome_indicador, tc, vc, unit=unit, category=cat)
            cards_html.append(card_wrap(
                fig=fig, title=nome_indicador, url=url, status=status,
                kpi_val=_format_kpi(kpi_val, unit), kpi_unit=unit, kpi_delta=kpi_delta, chart_unit=unit,
                status_color=status_color, status_label=status_label, tab_cat=tab_cat,
                is_fallback=is_fallback, defasagem_dias=defasagem,
                df_export=df if not is_fallback else None,
                dataset_slug=slug,
                vcol=vc or '',
            ))
            print(f" {status} {nome_indicador:<32} [{tab_cat.upper()}] [OK]")

    except Exception as e:
        falhas_indicadores.append(nome_indicador)
        print(f"   ❌ {nome_indicador}: {e}")

cards_html_string = "".join(cards_html)

print(f"\n✅ Processamento concluído: {len(dados_coletados)} indicadores.")
print(f"   • Reais: {total_indicadores - len(fallback_indicadores) - len(falhas_indicadores)}")
if fallback_indicadores:
    print(f"   • ⚠️ Simulados: {len(fallback_indicadores)} → {', '.join(fallback_indicadores)}")
if falhas_indicadores:
    print(f"   • ❌ Falharam: {len(falhas_indicadores)} → {', '.join(falhas_indicadores)}")

🚀 Iniciando coleta — 31 indicadores, 30 dias

   ⬇ ear-diario-por-subsistema
   ⬇ ena-diario-por-subsistema
   ⬇ carga-energia
   ⬇ cmo-semanal
   ⬇ cmo-semi-horario
   ⬇ intercambio-nacional
   ✓ cmo-semanal (2.2s)
   ⬇ restricao_coff_fotovoltaica
   ✓ carga-energia (2.6s)
   ⬇ ear-diario-por-bacia
   ✓ ear-diario-por-subsistema (2.9s)
   ⬇ ear-diario-por-ree-reservatorio-equivalente-de-energia
   ✓ ena-diario-por-subsistema (3.0s)
   ⬇ ear-diario-por-reservatorio
   ✓ ear-diario-por-ree-reservatorio-equivalente-de-energia (5.9s)
   ⬇ ena-diario-por-ree-reservatorio-equivalente-de-energia
   ✓ ear-diario-por-bacia (8.3s)
   ⬇ ena-diario-por-reservatorio
   ✓ ena-diario-por-ree-reservatorio-equivalente-de-energia (4.3s)
   ⬇ dados-hidrologicos-res
   ✓ intercambio-nacional (21.6s)
   ⬇ cvu-usitermica
   ✓ cmo-semi-horario (26.1s)
   ⬇ ccee-pld-media-diaria
   ✓ cvu-usitermica (5.1s)
   ⬇ programacao_x_previsao
   ⚠ PLD CCEE: acesso automatizado bloqueado pela CCEE (403); usando cópia o

In [ ]:
import json
import base64
from pathlib import Path
from IPython.display import HTML

# Gerar o template HTML final
# ==============================================================
# LOGIN LOCAL EDITÁVEL — altere somente estas duas linhas quando quiser.
# Observação: como o resultado é um HTML local estático, as credenciais
# ficam embutidas no navegador e NÃO equivalem a autenticação de produção.
LOGIN_USUARIO = 'admin'
LOGIN_SENHA = 'ons2026'
LOGIN_BG_PATH = '/home/ubuntu/upload/images(1).jpg'
try:
    login_bg_data = base64.b64encode(Path(LOGIN_BG_PATH).read_bytes()).decode('ascii')
    login_bg_url = 'data:image/jpeg;base64,' + login_bg_data
except Exception:
    login_bg_url = ''
LOGIN_USUARIO_JS = json.dumps(LOGIN_USUARIO, ensure_ascii=False)
LOGIN_SENHA_JS = json.dumps(LOGIN_SENHA, ensure_ascii=False)

ccee_modules_html = '<div class="ccee-modulos" style="margin-top:26px;">\n<h2 style="color:var(--accent); margin:0 0 12px;">Módulos CCEE recomendados</h2>\n<p style="color:var(--text-muted); margin:0 0 16px; font-size:13px;">Fontes públicas e áreas priorizadas para acompanhamento do mercado. Os cards PLD com dados e filtros integrados ficam apresentados abaixo.</p>\n<div class="ccee-module-grid">\n  <a class="ccee-module" href="https://www.ccee.org.br/precos/painel-precos" target="_blank"><strong>Preços e PLD</strong><span>PLD horário, diário, semanal e mensal por submercado.</span><em>Fonte oficial CCEE ↗</em></a>\n  <a class="ccee-module" href="https://www.ccee.org.br/dados-e-analises/dados-geracao" target="_blank"><strong>Geração</strong><span>Geração por usina, fonte, estado, submercado e MRE.</span><em>Próxima integração ↗</em></a>\n  <a class="ccee-module" href="https://www.ccee.org.br/consumo" target="_blank"><strong>Consumo</strong><span>Consumo por região, ambiente de contratação e atividade.</span><em>Próxima integração ↗</em></a>\n  <a class="ccee-module" href="https://www.ccee.org.br/dados-e-analises/mercado-quinzenal" target="_blank"><strong>Mercado Quinzenal</strong><span>Medição prévia, posição contratual e análises regionais.</span><em>Fonte oficial CCEE ↗</em></a>\n  <a class="ccee-module" href="https://www.ccee.org.br/contabilizacao" target="_blank"><strong>Contabilização e liquidação</strong><span>MCP, ESS, MRE, exposições, penalidades e recontabilização.</span><em>Próxima integração ↗</em></a>\n  <a class="ccee-module" href="https://www.ccee.org.br/dados-e-analises/seguranca-de-mercado" target="_blank"><strong>Segurança de mercado</strong><span>Garantias, inadimplência, concentração e balanço energético.</span><em>Fonte oficial CCEE ↗</em></a>\n  <a class="ccee-module" href="https://www.ccee.org.br/calendario" target="_blank"><strong>Agenda CCEE</strong><span>Operações, contas setoriais, liquidações e reuniões.</span><em>Fonte oficial CCEE ↗</em></a>\n  <a class="ccee-module" href="https://dadosabertos.ccee.org.br/" target="_blank"><strong>Dados Abertos</strong><span>Datasets, dicionários e recursos para automação.</span><em>API/Dataset CCEE ↗</em></a>\n</div></div>'

aneel_modules_html = '<div class="aneel-modulos" style="margin-top:26px;">\n<h2 style="color:var(--aneel-accent); margin:0 0 12px;">Módulos ANEEL recomendados</h2>\n<p style="color:var(--text-muted); margin:0 0 16px; font-size:13px;">Painéis, bases abertas e indicadores priorizados a partir do portal oficial da ANEEL. Os módulos informativos apontam para a fonte oficial e não inventam dados quando a base ainda não está integrada.</p>\n<div class="aneel-module-grid">\n  <a class="aneel-module" href="https://www.gov.br/aneel/pt-br/assuntos/tarifas" target="_blank"><strong>Tarifas e informações econômico-financeiras</strong><span>Tarifas de geração, transmissão, distribuição e comercialização, com foco em modicidade e sinalização ao mercado.</span><em>Fonte oficial ANEEL ↗</em></a>\n  <a class="aneel-module" href="https://app.powerbi.com/view?r=eyJrIjoiNGE3NjVmYjAtNDFkZC00MDY4LTliNTItMTVkZTU4NWYzYzFmIiwidCI6IjQwZDZmOWI4LWVjYTctNDZhMi05MmQ0LWVhNGU5YzAxNzBlMSIsImMiOjR9" target="_blank"><strong>SIGA — geração</strong><span>Usinas por fonte, estado, fase, capacidade, agente e sub-bacia.</span><em>Painel interativo ANEEL ↗</em></a>\n  <a class="aneel-module" href="https://app.powerbi.com/view?r=eyJrIjoiMjIxMDFlNjAtOGEyNC00Y2EwLTlmMzUtOWJjN2I2MTEzYzM0IiwidCI6IjQwZDZmOWI4LWVjYTctNDZhMi05MmQ0LWVhNGU5YzAxNzBlMSIsImMiOjR9" target="_blank"><strong>Outorgas de geração</strong><span>Empreendimentos, agentes, CEG, tipo de ato, fonte, data e localização.</span><em>Painel interativo ANEEL ↗</em></a>\n  <a class="aneel-module" href="https://portalrelatorios.aneel.gov.br/Ralie" target="_blank"><strong>Ralie — expansão da geração</strong><span>Acompanhamento de futuras usinas, atos administrativos e previsão de entrada em operação.</span><em>Relatório ANEEL ↗</em></a>\n  <a class="aneel-module" href="https://www.gov.br/aneel/pt-br/assuntos/distribuicao/qualidade-do-fornecimento-de-energia-eletrica" target="_blank"><strong>Qualidade — DEC e FEC</strong><span>Continuidade coletiva por distribuidora e conjunto de unidades consumidoras.</span><em>Fonte oficial ANEEL ↗</em></a>\n  <a class="aneel-module" href="https://www.gov.br/aneel/pt-br/centrais-de-conteudos/relatorios-e-indicadores/distribuicao/relatorios-distribuicao" target="_blank"><strong>DIC, FIC, DMIC e compensações</strong><span>Indicadores individuais, violações de limites e compensações ao consumidor.</span><em>Relatórios ANEEL ↗</em></a>\n  <a class="aneel-module" href="https://www.gov.br/aneel/pt-br/centrais-de-conteudos/relatorios-e-indicadores/micro-e-minigeracao-distribuida" target="_blank"><strong>MMGD</strong><span>Conexões de micro e minigeração distribuída e acompanhamento da expansão.</span><em>Relatório ANEEL ↗</em></a>\n  <a class="aneel-module" href="https://dadosabertos.aneel.gov.br/dataset/conta-desenvolvimento-energetico-cde-custeio-dos-beneficios-tarifarios" target="_blank"><strong>CDE e benefícios tarifários</strong><span>Base pública para acompanhamento da Conta de Desenvolvimento Energético.</span><em>Dados Abertos ANEEL ↗</em></a>\n  <a class="aneel-module" href="https://dadosabertos.aneel.gov.br/dataset/indicadores-coletivos-de-continuidade-dec-e-fec" target="_blank"><strong>Dados Abertos — DEC/FEC</strong><span>Limites e valores apurados dos indicadores coletivos em formato processável.</span><em>Dataset ANEEL ↗</em></a>\n  <a class="aneel-module" href="https://www.gov.br/aneel/pt-br/centrais-de-conteudos/procedimentos-regulatorios/prodist" target="_blank"><strong>PRODIST e qualidade da tensão</strong><span>Referências regulatórias para continuidade, tensão e atendimento ao consumidor.</span><em>Regulação ANEEL ↗</em></a>\n</div></div>'

epe_elec_html = '<div class="epe-modulos"><h2 style="color:#00A6A6; margin:0 0 12px;">EPE — Energia Elétrica</h2><p class="epe-intro">Dados, painéis e publicações oficiais sobre consumo, mercado, oferta e infraestrutura elétrica.</p><div class="aneel-module-grid">\n<a class="aneel-module epe-elec" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/consumo-de-energia-eletrica" target="_blank"><strong>Consumo mensal</strong><span>Série desde 2004 por classe, região, subsistema, UF e ambiente cativo ou livre.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-elec" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/anuario-estatistico-de-energia-eletrica" target="_blank"><strong>Anuário Estatístico de Energia Elétrica</strong><span>Estatísticas anuais da cadeia elétrica com recortes regionais e setoriais.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-elec" href="https://www.epe.gov.br/pt/areas-de-atuacao/energia-eletrica/consumo-de-energia-eletrica/painel-de-consumo-historico-de-energia-eletrica-desde-1970" target="_blank"><strong>Consumo histórico desde 1970</strong><span>Histórico de eletricidade com séries BEN e Eletrobras.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-elec" href="https://www.epe.gov.br/pt/areas-de-atuacao/energia-eletrica/consumo-de-energia-el%C3%A9trica/comissao-permanente-de-analise-e-acompanhamento-do-mercado-de-energia-eletrica-copam" target="_blank"><strong>Painel COPAM</strong><span>Consumo e número de consumidores por classe, região e subsistema.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-elec" href="https://www.epe.gov.br/pt/areas-de-atuacao/energia-eletrica/consumo-de-energia-el%C3%A9trica/mercado-de-distribuicao" target="_blank"><strong>Mercado de distribuição</strong><span>MMGD injetada e perdas na distribuição.</span><em>Fonte oficial EPE ↗</em></a></div></div>'
epe_plan_html = '<div class="epe-modulos"><h2 style="color:#7C3AED; margin:0 0 12px;">EPE — Planejamento Energético</h2><p class="epe-intro">Planos, cenários, demanda, expansão e parâmetros do planejamento energético nacional.</p><div class="aneel-module-grid">\n<a class="aneel-module epe-plan" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/plano-decenal-de-expansao-de-energia-2035" target="_blank"><strong>PDE 2035</strong><span>Expansão integrada do setor de energia no horizonte 2026–2035, com dados brutos e cadernos.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-plan" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/Plano-Nacional-de-Energia-2050" target="_blank"><strong>PNE 2050</strong><span>Recomendações e diretrizes para a estratégia energética de longo prazo.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-plan" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/plano-nacional-de-energia-2055" target="_blank"><strong>PNE 2055</strong><span>Cenários de longo prazo, transição energética e futuros possíveis.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-plan" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/balanco-energetico-nacional-ben" target="_blank"><strong>BEN e séries históricas</strong><span>Oferta, transformação, importação, exportação e consumo desde 1970.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-plan" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/ferramentas-interativas" target="_blank"><strong>Ferramentas interativas</strong><span>Hub de painéis para BEN, planos, oferta, consumo e investimentos.</span><em>Fonte oficial EPE ↗</em></a></div></div>'
epe_oil_html = '<div class="epe-modulos"><h2 style="color:#D97706; margin:0 0 12px;">EPE — Petróleo, Gás e Biocombustíveis</h2><p class="epe-intro">Estudos e perspectivas oficiais para combustíveis líquidos, gás natural e biocombustíveis.</p><div class="aneel-module-grid">\n<a class="aneel-module epe-oil" href="https://www.epe.gov.br/pt/areas-de-atuacao/petroleo-gas-e-biocombustiveis" target="_blank"><strong>Área de petróleo, gás e biocombustíveis</strong><span>Análises e projeções de médio e longo prazo da EPE.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-oil" href="https://www.epe.gov.br/pt/areas-de-atuacao/petroleo-gas-e-biocombustiveis/abastecimento-de-petr%C3%B3leo-e-derivados" target="_blank"><strong>Abastecimento de derivados</strong><span>Demanda, preços e infraestrutura de abastecimento.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-oil" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/plano-decenal-de-expansao-de-energia-2035" target="_blank"><strong>Produção de petróleo e gás</strong><span>Projeções de produção e oferta no planejamento decenal.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-oil" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/ferramentas-interativas" target="_blank"><strong>Biocombustíveis e gás natural</strong><span>Ferramentas e estudos sobre etanol, biodiesel, biometano e gasodutos.</span><em>Fonte oficial EPE ↗</em></a></div></div>'
epe_transition_html = '<div class="epe-modulos"><h2 style="color:#059669; margin:0 0 12px;">EPE — Transição Energética e Mapas</h2><p class="epe-intro">Mapas, tecnologias, biodiversidade, inovação, financiamento e transição energética.</p><div class="aneel-module-grid">\n<a class="aneel-module epe-trans" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/webmap-epe" target="_blank"><strong>WEBMAP EPE</strong><span>Camadas de usinas, linhas, subestações, combustíveis, gás, petróleo e ambiente.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-trans" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/biodivepe-biodiversidade-no-planejamento-de-projetos-de-energia" target="_blank"><strong>BiodivEPE</strong><span>Suporte geográfico à redução de externalidades sobre a biodiversidade.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-trans" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/plataforma-inova-e-panorama-dos-investimentos-de-inovacao-em-energia-no-brasil" target="_blank"><strong>inova-e e InvesTE</strong><span>Investimentos em inovação e financiamento da transição energética.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-trans" href="https://dashboard.epe.gov.br/apps/boletimeol/" target="_blank"><strong>Energia eólica</strong><span>Boletins, parques onshore, potencial e análises de incertezas e perdas.</span><em>Fonte oficial EPE ↗</em></a>\n<a class="aneel-module epe-trans" href="https://www.epe.gov.br/pt/publicacoes-dados-abertos/ferramentas-interativas" target="_blank"><strong>Hidrogênio e cidades sustentáveis</strong><span>Ferramentas para novas tecnologias, mobilidade e sustentabilidade.</span><em>Fonte oficial EPE ↗</em></a></div></div>'
cnpe_html = '<div class="epe-modulos"><h2 style="color:#DC2626; margin:0 0 12px;">CNPE — Conselho Nacional de Política Energética</h2><p class="epe-intro">Órgão de assessoramento do Presidente da República para formulação de políticas e diretrizes de energia.</p><div class="aneel-module-grid">\n<a class="aneel-module gov-cnpe" href="https://www.gov.br/mme/pt-br/assuntos/conselhos-e-comites/cnpe" target="_blank"><strong>Portal e competências</strong><span>Composição, atribuições, princípios e objetivos da política energética nacional.</span><em>Fonte oficial MME ↗</em></a>\n<a class="aneel-module gov-cnpe" href="https://www.gov.br/mme/pt-br/assuntos/conselhos-e-comites/arquivos/conselhos-e-comites/Resoluo_CNPE_14_2019.pdf" target="_blank"><strong>Resoluções CNPE</strong><span>Atos e diretrizes normativas para políticas e planejamento energético.</span><em>Documento oficial ↗</em></a>\n<a class="aneel-module gov-cnpe" href="https://www.gov.br/mme/pt-br/assuntos/conselhos-e-comites/arquivos/conselhos-e-comites/Port_CNPE_1_2019.pdf" target="_blank"><strong>Estrutura e funcionamento</strong><span>Portarias, estrutura institucional e regras de funcionamento do Conselho.</span><em>Documento oficial ↗</em></a>\n<a class="aneel-module gov-cnpe" href="https://www.planalto.gov.br/ccivil_03/leis/l9478.htm" target="_blank"><strong>Marco legal</strong><span>Lei nº 9.478/1997 e referências legais das atribuições do CNPE.</span><em>Planalto ↗</em></a></div></div>'
cmse_html = '<div class="epe-modulos"><h2 style="color:#2563EB; margin:0 0 12px;">CMSE — Comitê de Monitoramento do Setor Elétrico</h2><p class="epe-intro">Acompanha permanentemente a continuidade, a segurança do suprimento e as condições de atendimento dos setores de energia.</p><div class="aneel-module-grid">\n<a class="aneel-module gov-cmse" href="https://www.gov.br/mme/pt-br/assuntos/conselhos-e-comites/cmse" target="_blank"><strong>Competências e composição</strong><span>Participação de MME, ANEEL, ANP, CCEE, EPE e ONS, com atribuições de monitoramento integrado.</span><em>Fonte oficial MME ↗</em></a>\n<a class="aneel-module gov-cmse" href="https://www.gov.br/mme/pt-br/assuntos/conselhos-e-comites/cmse/atas" target="_blank"><strong>Atas das reuniões</strong><span>Registro das deliberações, análises de abastecimento e recomendações do Comitê.</span><em>Fonte oficial MME ↗</em></a>\n<a class="aneel-module gov-cmse" href="https://www.gov.br/mme/pt-br/assuntos/conselhos-e-comites/cmse/atas/2026-2" target="_blank"><strong>Atas CMSE 2026</strong><span>Acompanhamento das reuniões mais recentes e documentos publicados pelo MME.</span><em>Fonte oficial MME ↗</em></a>\n<a class="aneel-module gov-cmse" href="https://www.gov.br/mme/pt-br/assuntos/noticias" target="_blank"><strong>Comunicados e medidas</strong><span>Notícias e decisões relacionadas à segurança do suprimento eletroenergético.</span><em>Fonte oficial MME ↗</em></a></div></div>'
megawhat_html = '<div class="epe-modulos"><h2 style="color:#9333EA; margin:0 0 12px;">Minuto MegaWhat</h2><p class="epe-intro">Conteúdo jornalístico em áudio para acompanhar acontecimentos, preços, regulação e mercado de energia.</p><div class="aneel-module-grid">\n<a class="aneel-module megawhat" href="https://megawhat.uol.com.br/podcasts/minutomega/" target="_blank"><strong>Lista de episódios</strong><span>Últimos episódios e análises rápidas sobre o setor elétrico e energético.</span><em>Fonte MegaWhat ↗</em></a>\n<a class="aneel-module megawhat" href="https://open.spotify.com/show/2nJDlUvgT4oNz7ibV1tFdb" target="_blank"><strong>MinutoMega no Spotify</strong><span>Acesso ao feed de áudio e aos episódios distribuídos pela plataforma.</span><em>Fonte MegaWhat/Spotify ↗</em></a>\n<a class="aneel-module megawhat" href="https://www.youtube.com/channel/UC-OLmHwxwzeM4dO79kYX-tg" target="_blank"><strong>MegaWhat no YouTube</strong><span>Vídeos, entrevistas e conteúdos complementares sobre energia.</span><em>Canal oficial ↗</em></a>\n<a class="aneel-module megawhat" href="https://megawhat.uol.com.br/" target="_blank"><strong>Portal MegaWhat</strong><span>Notícias, análises e cobertura de política, mercado e infraestrutura energética.</span><em>Fonte MegaWhat ↗</em></a></div></div>'
html_template = f"""<!DOCTYPE html><html><head><meta charset='utf-8'><script>{get_plotlyjs()}</script><meta name='viewport' content='width=device-width, initial-scale=1'><style>
:root {{
    --navy:#082a63; --navy-2:#0d3b85; --accent:#2563eb; --accent-2:#14b8a6;
    --bg-page:#f5f7fb; --bg-card:#ffffff; --text-main:#17213a; --text-muted:#6b7891;
    --border:#e3e8f0; --shadow:0 8px 24px rgba(17,44,88,.08); --danger:#c0392b;
}}
body.dark {{
    --navy:#071b40; --navy-2:#0d2b5d; --accent:#60a5fa; --accent-2:#2dd4bf;
    --bg-page:#0e172a; --bg-card:#13213a; --text-main:#e8eefb; --text-muted:#9fb0ca;
    --border:#263858; --shadow:0 10px 28px rgba(0,0,0,.28);
}}
* {{ box-sizing:border-box; }}
html,body {{ min-height:100%; margin:0; }}
body {{ min-height:100vh; margin:0; background:var(--bg-page); color:var(--text-main); font-family:'Segoe UI',Inter,Arial,sans-serif; transition:background .25s,color .25s; }}
a {{ color:inherit; }}
#login-screen {{ position:relative; z-index:9999; display:flex; align-items:center; justify-content:center; width:100%; min-height:760px; height:100vh; padding:32px 20px; background:linear-gradient(135deg,rgba(245,247,251,.88),rgba(224,232,247,.80)),url('{login_bg_url}') center/cover no-repeat; }}
body.dark #login-screen {{ background:linear-gradient(135deg,rgba(7,27,64,.92),rgba(14,23,42,.88)),url('{login_bg_url}') center/cover no-repeat; }}
#login-screen[style*="display: none"] {{ display:none !important; }}
.login-box {{ width:min(560px,calc(100vw - 48px)); padding:46px 48px; border-radius:24px; background:rgba(255,255,255,.93); border:1px solid rgba(255,255,255,.8); box-shadow:0 20px 70px rgba(8,42,99,.22); text-align:center; }}
body.dark .login-box {{ background:rgba(19,33,58,.96); border-color:var(--border); box-shadow:0 20px 70px rgba(0,0,0,.45); }}
.login-box h2 {{ margin:0 0 12px; color:var(--navy); font-size:clamp(24px,3vw,34px); }}
body.dark .login-box h2 {{ color:#8ec5ff; }}
.login-box p {{ color:var(--text-muted); font-size:15px; margin:0 0 26px; }}
.login-field {{ width:100%; margin:10px 0; padding:15px 16px; border-radius:10px; border:1px solid var(--border); background:var(--bg-card); color:var(--text-main); font-size:15px; outline:none; }}
.login-submit {{ width:100%; margin-top:16px; padding:15px; border:0; border-radius:10px; color:#fff; font-weight:700; cursor:pointer; background:linear-gradient(90deg,var(--navy),var(--accent)); }}
.login-error {{ min-height:20px; margin-top:12px; color:#e05252; font-size:13px; }}
#app-shell {{ display:none; }}
.app-layout {{ min-height:100vh; display:flex; }}
.sidebar {{ position:sticky; top:0; width:238px; min-width:238px; height:100vh; display:flex; flex-direction:column; padding:24px 16px 18px; background:linear-gradient(180deg,var(--navy),#0a3477); color:#fff; }}
.sidebar-toggle {{ width:100%; display:flex; align-items:center; gap:10px; margin:0 0 12px; padding:9px 11px; border:1px solid rgba(255,255,255,.18); border-radius:9px; background:rgba(255,255,255,.08); color:#fff; cursor:pointer; font-size:11px; font-weight:700; text-align:left; }}
.sidebar-toggle:hover {{ background:rgba(255,255,255,.16); }}
.sidebar-reopen {{ position:fixed; left:10px; top:18px; z-index:20; display:none; align-items:center; justify-content:center; width:38px; height:38px; border:1px solid var(--border); border-radius:10px; background:var(--bg-card); color:var(--accent); box-shadow:var(--shadow); cursor:pointer; font-size:20px; font-weight:800; }}
body.sidebar-collapsed .sidebar {{ width:72px; min-width:72px; padding:20px 10px 18px; }}
body.sidebar-collapsed .brand {{ justify-content:center; padding:0 0 24px; }}
body.sidebar-collapsed .brand strong, body.sidebar-collapsed .brand small, body.sidebar-collapsed .nav-section-label, body.sidebar-collapsed .tab-btn span:not(.nav-icon), body.sidebar-collapsed .sidebar-footer, body.sidebar-collapsed .sidebar-toggle-label {{ display:none; }}
body.sidebar-collapsed .tab-btn {{ justify-content:center; padding:12px 8px; }}
body.sidebar-collapsed .sidebar-toggle {{ justify-content:center; padding:9px 8px; }}
body.sidebar-collapsed .sidebar-toggle .nav-icon {{ transform:rotate(180deg); }}
body.sidebar-collapsed .sidebar-reopen {{ display:flex; }}
.brand {{ display:flex; align-items:center; gap:11px; padding:0 10px 30px; }}
.brand-mark {{ width:34px; height:34px; display:grid; place-items:center; border-radius:10px; background:linear-gradient(135deg,#38bdf8,#14b8a6); color:#fff; font-size:20px; font-weight:800; box-shadow:0 5px 14px rgba(20,184,166,.28); }}
.brand strong {{ display:block; font-size:15px; line-height:1.18; letter-spacing:.1px; }}
.brand small {{ display:block; margin-top:3px; color:#b9d4ff; font-size:10px; }}
.sidebar-nav {{ display:flex; flex-direction:column; gap:5px; }}
.nav-section-label {{ padding:14px 12px 7px; color:#91b5eb; font-size:10px; font-weight:700; text-transform:uppercase; letter-spacing:1px; }}
.tab-btn {{ width:100%; display:flex; align-items:center; gap:11px; padding:11px 13px; border:1px solid transparent; border-radius:9px; background:transparent; color:#d8e6ff; cursor:pointer; text-align:left; font-size:13px; font-weight:600; transition:all .2s; }}
.tab-btn:hover {{ background:rgba(255,255,255,.10); color:#fff; }}
.tab-btn.active {{ background:linear-gradient(90deg,#1e65d6,#2e7be6); border-color:rgba(255,255,255,.16); color:#fff; box-shadow:0 5px 14px rgba(0,0,0,.15); }}
.nav-icon {{ width:18px; text-align:center; font-size:15px; opacity:.95; }}
.sidebar-footer {{ margin-top:auto; padding:16px 11px 4px; color:#b9d4ff; font-size:11px; line-height:1.5; border-top:1px solid rgba(255,255,255,.12); }}
.live-dot {{ display:inline-block; width:8px; height:8px; margin-right:6px; border-radius:50%; background:#22c55e; box-shadow:0 0 0 3px rgba(34,197,94,.15); }}
.main-content {{ flex:1; min-width:0; padding:28px 30px 22px; }}
.topbar {{ display:flex; align-items:flex-start; justify-content:space-between; gap:22px; margin-bottom:20px; }}
.eyebrow {{ margin:0 0 5px; color:var(--accent); font-size:10px; font-weight:800; letter-spacing:1.15px; text-transform:uppercase; }}
h1 {{ margin:0; color:var(--text-main); font-size:28px; line-height:1.15; letter-spacing:-.4px; }}
.top-subtitle {{ margin:7px 0 0; color:var(--text-muted); font-size:13px; }}
.topbar-actions {{ display:flex; align-items:center; justify-content:flex-end; gap:9px; flex-wrap:wrap; }}
.period-badge,.updated-badge {{ display:inline-flex; align-items:center; gap:7px; padding:9px 12px; border:1px solid var(--border); border-radius:9px; background:var(--bg-card); color:var(--text-muted); font-size:12px; white-space:nowrap; }}
.updated-badge::before {{ content:''; width:7px; height:7px; border-radius:50%; background:#22c55e; }}
.btn-theme,.header-export {{ padding:9px 13px; border:1px solid var(--border); border-radius:9px; background:var(--bg-card); color:var(--text-main); cursor:pointer; font-size:12px; font-weight:700; }}
.header-export {{ border-color:var(--accent); color:var(--accent); }}
.btn-theme:hover,.header-export:hover {{ background:var(--accent); color:#fff; }}
.filter-bar {{ display:flex; align-items:center; gap:14px; flex-wrap:wrap; padding:12px 15px; margin-bottom:20px; border:1px solid var(--border); border-radius:12px; background:var(--bg-card); box-shadow:var(--shadow); }}
.filter-item {{ display:flex; flex-direction:column; gap:3px; min-width:145px; }}
.filter-item span {{ color:var(--text-muted); font-size:10px; text-transform:uppercase; letter-spacing:.6px; font-weight:700; }}
.filter-item strong {{ color:var(--text-main); font-size:13px; }}
.filter-note {{ flex:1; min-width:190px; color:var(--text-muted); font-size:11px; }}
.section-heading {{ display:flex; align-items:flex-end; justify-content:space-between; gap:12px; margin:2px 0 14px; }}
.section-heading h2 {{ margin:0; color:var(--text-main); font-size:18px; }}
.section-heading p {{ margin:4px 0 0; color:var(--text-muted); font-size:12px; }}
.section-heading .section-status {{ color:var(--accent-2); font-size:11px; font-weight:700; }}
.tab-panel {{ display:none; }}
.tab-panel.active {{ display:block; }}
.kpi-grid {{ display:grid; grid-template-columns:repeat(6,minmax(0,1fr)); gap:12px; margin-bottom:18px; }}
.kpi-tile {{ min-height:104px; padding:14px; border:1px solid var(--border); border-radius:12px; background:var(--bg-card); box-shadow:var(--shadow); }}
.kpi-label {{ display:block; color:var(--text-muted); font-size:11px; font-weight:600; }}
.kpi-value {{ display:block; margin-top:7px; color:var(--text-main); font-size:19px; font-weight:800; letter-spacing:-.2px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }}
.kpi-foot {{ display:block; margin-top:8px; color:var(--text-muted); font-size:10px; }}
.kpi-accent {{ display:block; width:28px; height:3px; margin-top:-2px; margin-bottom:8px; border-radius:3px; background:var(--accent); }}
.kpi-tile:nth-child(2) .kpi-accent {{ background:#b06d5f; }} .kpi-tile:nth-child(3) .kpi-accent {{ background:#14b8a6; }} .kpi-tile:nth-child(4) .kpi-accent {{ background:#8b5cf6; }} .kpi-tile:nth-child(5) .kpi-accent {{ background:#22c55e; }} .kpi-tile:nth-child(6) .kpi-accent {{ background:#f59e0b; }}
.overview-feature {{ display:grid; grid-template-columns:minmax(0,1fr) 245px; gap:0; margin-bottom:18px; border:1px solid var(--border); border-radius:14px; background:var(--bg-card); box-shadow:var(--shadow); overflow:hidden; }}
.overview-cvu-card {{ min-width:0; padding:18px 20px 12px; }}
.overview-card-header {{ display:flex; align-items:flex-start; justify-content:space-between; gap:12px; margin-bottom:8px; }}
.overview-card-header h2 {{ margin:0; color:var(--text-main); font-size:18px; }}
.overview-card-header p {{ margin:5px 0 0; color:#a2675a; font-size:12px; font-weight:700; letter-spacing:.2px; }}
.overview-chart-note {{ color:var(--text-muted); font-size:11px; text-align:right; }}
#overview-cvu-summary {{ margin:8px 0 4px; }}
#overview-cvu-summary .cvu-region-summary-grid {{ margin:0; }}
.overview-cvu-chart {{ min-height:405px; width:100%; }}
.overview-detail {{ padding:20px 17px; border-left:1px solid var(--border); background:linear-gradient(180deg,rgba(37,99,235,.04),transparent); }}
.overview-detail h3 {{ margin:0 0 18px; font-size:14px; color:var(--text-main); }}
.detail-item {{ padding:13px 0; border-bottom:1px solid var(--border); }}
.detail-item:last-child {{ border-bottom:0; }}
.detail-item span {{ display:block; color:var(--text-muted); font-size:10px; text-transform:uppercase; letter-spacing:.6px; font-weight:700; }}
.detail-item strong {{ display:block; margin-top:5px; color:var(--text-main); font-size:15px; }}
.detail-source {{ display:inline-block; margin-top:17px; color:var(--accent); font-size:11px; font-weight:700; text-decoration:none; }}
.detail-button {{ width:100%; margin-top:16px; padding:9px 10px; border:1px solid var(--accent); border-radius:8px; background:transparent; color:var(--accent); cursor:pointer; font-size:11px; font-weight:700; }}
.overview-lower-grid {{ display:grid; grid-template-columns:repeat(3,minmax(0,1fr)); gap:13px; margin-bottom:20px; }}
.overview-lower-card {{ padding:15px 16px; border:1px solid var(--border); border-radius:12px; background:var(--bg-card); box-shadow:var(--shadow); }}
.overview-lower-card h3 {{ margin:0 0 7px; color:var(--text-main); font-size:13px; }}
.overview-lower-card p {{ margin:0; color:var(--text-muted); font-size:11px; line-height:1.5; }}
.overview-lower-card strong {{ color:var(--accent); font-size:22px; }}
.grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(380px,1fr)); gap:16px; }}
.grid#grid-ons {{ grid-template-columns:1fr; }}
.grid#grid-ons > .card-item {{ width:100%; box-sizing:border-box; }}
.card-item {{ min-width:0; padding:17px; border:1px solid var(--border); border-left:4px solid var(--card-accent,var(--accent)); border-radius:13px; background:var(--bg-card); box-shadow:var(--shadow); transition:transform .18s,box-shadow .18s,border-color .18s; }}
.card-item:hover {{ transform:translateY(-2px); box-shadow:0 13px 30px rgba(17,44,88,.13); }}
.card-item.cvu-consolidado {{ grid-column:1/-1; border-top:3px solid #a96e5f; border-left-color:#a96e5f; }}
.card-item.pld-card {{ border-top:3px solid var(--card-accent,var(--accent)); }}
.card-header {{ display:flex; justify-content:space-between; align-items:flex-start; gap:12px; }}
.card-title-wrap {{ display:flex; align-items:center; gap:8px; min-width:0; }}
.card-title {{ color:var(--text-main); cursor:pointer; font-size:14px; line-height:1.25; }}
.card-title:hover {{ color:var(--accent); }}
.status-dot {{ width:8px; min-width:8px; height:8px; border-radius:50%; margin-top:2px; }}
.status-dot.success {{ background:#19a974; box-shadow:0 0 0 3px rgba(25,169,116,.12); }}
.status-dot.warning {{ background:#f59e0b; box-shadow:0 0 0 3px rgba(245,158,11,.12); }}
.data-badge {{ padding:3px 6px; border-radius:999px; background:#fff4e5; color:#9a5b00; font-size:9px; font-weight:700; white-space:nowrap; }}
.card-actions {{ display:flex; align-items:center; justify-content:flex-end; flex-wrap:wrap; gap:5px; }}
.source-link,.download-link {{ display:inline-flex; align-items:center; padding:5px 7px; border:1px solid var(--border); border-radius:6px; text-decoration:none; font-size:10px; font-weight:700; white-space:nowrap; }}
.source-link {{ color:var(--text-muted); }}
.download-excel {{ border-color:#1c8c5e; color:#147346; background:rgba(28,140,94,.05); }}
.download-csv {{ border-color:#2d78d2; color:#1d5fae; background:rgba(45,120,210,.05); }}
.source-link:hover,.download-link:hover {{ border-color:var(--accent); color:var(--accent); }}
.card-kpi-value {{ margin:13px 0 2px; color:var(--text-main); font-size:22px; font-weight:800; letter-spacing:-.3px; }}
.card-kpi-unit {{ color:var(--text-muted); font-size:12px; font-weight:600; }}
.card-meta {{ display:flex; align-items:center; flex-wrap:wrap; gap:8px; min-height:18px; }}
.card-delta,.freshness,.status-label {{ font-size:10px; font-weight:700; }}
.freshness {{ color:var(--text-muted); }} .status-label {{ color:var(--accent); }}
.card-controls {{ display:flex; align-items:flex-end; gap:8px; flex-wrap:wrap; margin:14px 0 10px; padding:10px; border:1px solid var(--border); border-radius:9px; background:var(--bg-page); }}
.control-field {{ display:flex; flex-direction:column; gap:4px; color:var(--text-muted); font-size:10px; font-weight:700; }}
.control-field input,.control-field select,.usina-select {{ min-height:30px; padding:5px 8px; border:1px solid var(--border); border-radius:6px; background:var(--bg-card); color:var(--text-main); font-size:11px; }}
.btn-atualizar {{ min-height:30px; padding:5px 12px; border:0; border-radius:6px; background:var(--accent); color:#fff; cursor:pointer; font-size:11px; font-weight:700; }}
.btn-atualizar:hover {{ filter:brightness(1.08); }}
.msg-status {{ color:var(--text-muted); font-size:10px; }}
.cvu-region-summary-grid {{ display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:8px; margin:10px 0 12px; }}
.cvu-region-summary {{ min-height:60px; display:flex; flex-direction:column; gap:4px; padding:9px 10px; border-radius:8px; background:var(--bg-page); }}
.cvu-region-summary span {{ color:var(--text-main); font-size:14px; font-weight:800; }}
.cvu-region-summary small,.cvu-help {{ color:var(--text-muted); font-size:10px; }}
.cvu-help {{ margin:-2px 0 9px; }}
.epe-modulos {{ margin-bottom:8px; }} .epe-intro {{ color:var(--text-muted); margin:0 0 16px; font-size:13px; }}
.aneel-module-grid,.ccee-module-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(240px,1fr)); gap:14px; }}
.aneel-module,.ccee-module {{ display:flex; flex-direction:column; gap:7px; padding:15px; border:1px solid var(--border); border-left:4px solid var(--accent); border-radius:12px; background:var(--bg-card); color:var(--text-main); text-decoration:none; box-shadow:var(--shadow); }}
.aneel-module:hover,.ccee-module:hover {{ transform:translateY(-2px); }}
.aneel-module strong,.ccee-module strong {{ color:var(--accent); font-size:14px; }} .aneel-module span,.ccee-module span {{ color:var(--text-muted); font-size:12px; line-height:1.4; }}
#painel-detalhe {{ display:none; margin-bottom:30px; }}
.detail-back {{ margin-bottom:16px; padding:8px 16px; border:1px solid var(--accent); border-radius:8px; background:var(--bg-card); color:var(--accent); cursor:pointer; font-weight:700; font-size:12px; }}
.detail-layout {{ display:grid; grid-template-columns:minmax(0,1fr) 340px; gap:20px; align-items:start; }}
#detalhe-grafico {{ min-height:400px; padding:14px; border:1px solid var(--border); border-radius:12px; background:var(--bg-card); }}
#detalhe-info {{ min-height:300px; padding:20px; border:1px solid var(--border); border-radius:12px; background:var(--bg-card); box-shadow:var(--shadow); }}
.footer {{ margin-top:25px; padding-top:15px; border-top:1px solid var(--border); color:var(--text-muted); text-align:center; font-size:10px; }}
    .external-portal-toolbar {{ display:flex; align-items:center; gap:12px; flex-wrap:wrap; margin-bottom:14px; padding:12px 14px; border:1px solid var(--border); border-radius:12px; background:var(--bg-card); box-shadow:var(--shadow); }}
    .external-portal-status {{ display:inline-flex; align-items:center; gap:7px; color:#16845b; font-size:11px; font-weight:700; }}
    .external-portal-status::before {{ content:''; width:8px; height:8px; border-radius:50%; background:#22c55e; box-shadow:0 0 0 3px rgba(34,197,94,.13); }}
    .external-portal-note {{ flex:1; min-width:220px; color:var(--text-muted); font-size:11px; }}
    .external-portal-link {{ display:inline-flex; align-items:center; padding:7px 10px; border:1px solid var(--accent); border-radius:7px; color:var(--accent); font-size:11px; font-weight:700; text-decoration:none; white-space:nowrap; }}
    .external-portal-link:hover {{ background:var(--accent); color:#fff; }}
    .external-portal-layout {{ display:grid; grid-template-columns:minmax(0,1fr) 250px; gap:15px; align-items:stretch; }}
    .external-portal-frame-card {{ min-height:760px; overflow:hidden; border:1px solid var(--border); border-radius:14px; background:var(--bg-card); box-shadow:var(--shadow); }}
    .external-portal-frame {{ display:block; width:100%; height:760px; border:0; background:var(--bg-card); }}
    .external-portal-info {{ padding:18px; border:1px solid var(--border); border-radius:14px; background:var(--bg-card); box-shadow:var(--shadow); }}
    .external-portal-info h3 {{ margin:0 0 12px; color:var(--text-main); font-size:15px; }}
    .external-portal-info p {{ color:var(--text-muted); font-size:11px; line-height:1.55; }}
    .external-portal-info-row {{ padding:12px 0; border-bottom:1px solid var(--border); }}
    .external-portal-info-row:last-of-type {{ border-bottom:0; }}
    .external-portal-info-row span {{ display:block; color:var(--text-muted); font-size:10px; font-weight:700; text-transform:uppercase; letter-spacing:.5px; }}
    .external-portal-info-row strong {{ display:block; margin-top:5px; color:var(--text-main); font-size:12px; }}
    .external-portal-warning {{ margin-top:15px; padding:10px; border-radius:8px; background:rgba(245,158,11,.10); color:#9a5b00; font-size:10px; line-height:1.45; }}
    @media (max-width:900px) {{ .external-portal-layout {{ grid-template-columns:1fr; }} .external-portal-frame-card {{ min-height:680px; }} .external-portal-frame {{ height:680px; }} }}
    @media (max-width:680px) {{ .external-portal-frame-card {{ min-height:580px; }} .external-portal-frame {{ height:580px; }} }}

    .weather-toolbar {{ display:flex; align-items:flex-end; gap:12px; flex-wrap:wrap; margin-bottom:14px; padding:12px 14px; border:1px solid var(--border); border-radius:12px; background:var(--bg-card); box-shadow:var(--shadow); }}
    .weather-control {{ display:flex; flex-direction:column; gap:4px; color:var(--text-muted); font-size:10px; font-weight:700; text-transform:uppercase; letter-spacing:.5px; }}
    .weather-control select {{ min-height:32px; min-width:142px; padding:6px 9px; border:1px solid var(--border); border-radius:7px; background:var(--bg-page); color:var(--text-main); font-size:12px; text-transform:none; letter-spacing:0; }}
    .weather-context {{ flex:1; min-width:200px; padding-bottom:8px; color:var(--text-muted); font-size:11px; }}
    .weather-open-link,.weather-info-source {{ color:var(--accent); font-size:11px; font-weight:700; text-decoration:none; white-space:nowrap; }}
    .weather-layout {{ display:grid; grid-template-columns:minmax(0,1fr) 275px; gap:15px; margin-bottom:15px; }}
    .weather-map-card {{ min-height:570px; overflow:hidden; border:1px solid #1f477d; border-radius:14px; background:#0a2858; box-shadow:var(--shadow); }}
    .weather-map-card iframe {{ display:block; width:100%; min-height:570px; height:100%; border:0; }}
    .weather-info-card {{ padding:18px; border:1px solid var(--border); border-radius:14px; background:var(--bg-card); box-shadow:var(--shadow); }}
    .weather-info-heading {{ display:flex; align-items:center; justify-content:space-between; gap:10px; }}
    .weather-info-card h3 {{ margin:0; color:var(--text-main); font-size:15px; }}
    .weather-info-card p {{ color:var(--text-muted); font-size:11px; line-height:1.5; }}
    .weather-live-dot {{ width:9px; height:9px; border-radius:50%; background:#22c55e; box-shadow:0 0 0 4px rgba(34,197,94,.13); }}
    .weather-info-row {{ padding:13px 0; border-bottom:1px solid var(--border); }}
    .weather-info-row span {{ display:block; color:var(--text-muted); font-size:10px; text-transform:uppercase; letter-spacing:.5px; font-weight:700; }}
    .weather-info-row strong {{ display:block; margin-top:5px; color:var(--text-main); font-size:13px; }}
    .weather-info-source {{ display:inline-block; margin-top:18px; }}
    .weather-bottom-grid {{ display:grid; grid-template-columns:repeat(3,minmax(0,1fr)); gap:13px; }}
    .weather-bottom-card {{ padding:15px 16px; border:1px solid var(--border); border-radius:12px; background:var(--bg-card); box-shadow:var(--shadow); }}
    .weather-bottom-card h3 {{ margin:0 0 7px; color:var(--text-main); font-size:13px; }}
    .weather-bottom-card p {{ margin:0; color:var(--text-muted); font-size:11px; line-height:1.5; }}
    .epe-modulos {{ margin-bottom:8px; }}
    body.dark .weather-map-card {{ border-color:#315d9a; }}
    @media (max-width:900px) {{ .weather-layout {{ grid-template-columns:1fr; }} .weather-map-card,.weather-map-card iframe {{ min-height:500px; }} }}
    @media (max-width:680px) {{ .weather-bottom-grid {{ grid-template-columns:1fr; }} .weather-map-card,.weather-map-card iframe {{ min-height:430px; }} }}

body.dark .card-item,body.dark .kpi-tile,body.dark .overview-feature,body.dark .overview-lower-card {{ background:var(--bg-card); }}
body.dark .download-excel {{ color:#64d89b; border-color:#2e9c6c; }} body.dark .download-csv {{ color:#8bbcff; border-color:#4b8fe5; }}
@media (max-width:1250px) {{ .kpi-grid {{ grid-template-columns:repeat(3,minmax(0,1fr)); }} }}
@media (max-width:900px) {{ .sidebar {{ width:76px; min-width:76px; padding:20px 10px; }} .brand {{ justify-content:center; padding:0 0 24px; }} .brand strong,.brand small,.nav-section-label,.tab-btn span:not(.nav-icon),.sidebar-footer {{ display:none; }} .tab-btn {{ justify-content:center; padding:12px 8px; }} .main-content {{ padding:22px 18px; }} .overview-feature {{ grid-template-columns:1fr; }} .overview-detail {{ border-left:0; border-top:1px solid var(--border); }} }}
@media (max-width:680px) {{ .topbar {{ flex-direction:column; }} .topbar-actions {{ justify-content:flex-start; }} .kpi-grid {{ grid-template-columns:repeat(2,minmax(0,1fr)); }} .overview-lower-grid {{ grid-template-columns:1fr; }} .grid {{ grid-template-columns:1fr; }} .cvu-region-summary-grid {{ grid-template-columns:repeat(2,minmax(0,1fr)); }} .card-header {{ flex-direction:column; }} .card-actions {{ justify-content:flex-start; }} .main-content {{ padding:18px 12px; }} h1 {{ font-size:23px; }} }}

/* Camada UX executiva adicionada sem alterar coleta, transformação ou gráficos. */
:focus-visible {{ outline:3px solid #F59E0B; outline-offset:2px; }}
.ux-toolbar {{ display:flex; align-items:end; gap:10px; flex-wrap:wrap; margin:0 0 16px; padding:12px 14px; border:1px solid var(--border); border-radius:12px; background:var(--bg-card); box-shadow:var(--shadow); }}
.ux-field {{ display:flex; flex-direction:column; gap:4px; min-width:145px; color:var(--text-muted); font-size:10px; font-weight:700; text-transform:uppercase; letter-spacing:.45px; }}
.ux-field input,.ux-field select {{ min-height:32px; padding:6px 9px; border:1px solid var(--border); border-radius:7px; background:var(--bg-page); color:var(--text-main); font-size:12px; text-transform:none; letter-spacing:0; }}
.ux-toolbar-actions {{ display:flex; gap:7px; align-items:center; margin-left:auto; }}
.ux-btn {{ min-height:32px; padding:6px 10px; border:1px solid var(--border); border-radius:7px; background:var(--bg-page); color:var(--text-main); cursor:pointer; font-size:11px; font-weight:700; }}
.ux-btn:hover {{ border-color:var(--accent); color:var(--accent); }}
.ux-btn-primary {{ background:var(--accent); border-color:var(--accent); color:#fff; }}
.ux-governance {{ display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:9px; margin:0 0 20px; }}
.ux-governance-item {{ padding:10px 12px; border:1px solid var(--border); border-radius:9px; background:var(--bg-card); }}
.ux-governance-item span {{ display:block; color:var(--text-muted); font-size:9px; font-weight:700; letter-spacing:.5px; text-transform:uppercase; }}
.ux-governance-item strong {{ display:block; margin-top:4px; color:var(--text-main); font-size:12px; }}
.ux-governance-item strong.ok {{ color:#16845b; }}
.card-item.is-favorite {{ border-color:#F59E0B; box-shadow:0 0 0 2px rgba(245,158,11,.16),var(--shadow); }}
.card-item.ux-hidden {{ display:none !important; }}
.ux-empty {{ display:none; padding:28px; border:1px dashed var(--border); border-radius:12px; color:var(--text-muted); text-align:center; background:var(--bg-card); }}
.compact-mode .card-item {{ padding:10px !important; }}
.compact-mode .grid {{ gap:10px !important; }}
@media (max-width:900px) {{ .ux-governance {{ grid-template-columns:repeat(2,minmax(0,1fr)); }} .ux-toolbar-actions {{ margin-left:0; }} }}
@media (max-width:680px) {{ .ux-governance {{ grid-template-columns:1fr; }} .ux-field {{ min-width:100%; }} .ux-toolbar-actions {{ width:100%; }} .ux-btn {{ flex:1; }} }}
</style></head><body>
<div id="login-screen" aria-label="Tela de acesso"><form class="login-box" onsubmit="return realizarLogin(event)"><h2>Acesso ao Dashboard</h2><p>Informe suas credenciais para continuar</p><input id="login-user" class="login-field" autocomplete="username" placeholder="Usuário" required><input id="login-pass" class="login-field" type="password" autocomplete="current-password" placeholder="Senha" required><button class="login-submit" type="submit">Entrar</button><div id="login-error" class="login-error"></div></form></div>
<div id="app-shell"><div class="app-layout">
<aside class="sidebar">
    <div class="brand"><div class="brand-mark">⚡</div><div><strong>Painel de Dados<br>do Setor Elétrico</strong><small>Monitoramento integrado</small></div></div>
    <button id="sidebar-toggle" class="sidebar-toggle" type="button" onclick="alternarMenuLateral()" aria-controls="sidebar-nav" aria-expanded="true"><span class="nav-icon">‹</span><span class="sidebar-toggle-label">Ocultar menu</span></button>
    <nav id="sidebar-nav" class="sidebar-nav" aria-label="Navegação principal">
        <div class="nav-section-label">Navegação</div>
        <button class="tab-btn active" data-tab="visao-geral" onclick="ativarAba('visao-geral')"><span class="nav-icon">▦</span><span>Visão Geral</span></button>
        <button class="tab-btn" data-tab="ons" onclick="ativarAba('ons')"><span class="nav-icon">⌁</span><span>Dados ONS</span></button>
        <button class="tab-btn" data-tab="ccee" onclick="ativarAba('ccee')"><span class="nav-icon">◉</span><span>Dados CCEE</span></button>
        <button class="tab-btn" data-tab="aneel" onclick="ativarAba('aneel')"><span class="nav-icon">▤</span><span>Dados ANEEL</span></button>
        <button class="tab-btn" data-tab="meteorologia" onclick="ativarAba('meteorologia')"><span class="nav-icon">☁</span><span>Meteorologia</span></button>
        <button class="tab-btn" data-tab="denergia" onclick="ativarAba('denergia')"><span class="nav-icon">◒</span><span>Denergia</span></button>
        <button class="tab-btn" data-tab="cammesa" onclick="ativarAba('cammesa')"><span class="nav-icon">▰</span><span>CAMMESA</span></button>
        <div class="nav-section-label">Fontes setoriais</div>
        <button class="tab-btn" data-tab="epe-elec" onclick="ativarAba('epe-elec')"><span class="nav-icon">◈</span><span>EPE Elétrica</span></button>
        <button class="tab-btn" data-tab="epe-plan" onclick="ativarAba('epe-plan')"><span class="nav-icon">⌂</span><span>EPE Planejamento</span></button>
        <button class="tab-btn" data-tab="epe-oil" onclick="ativarAba('epe-oil')"><span class="nav-icon">◉</span><span>EPE Petróleo/Gás</span></button>
        <button class="tab-btn" data-tab="epe-trans" onclick="ativarAba('epe-trans')"><span class="nav-icon">✦</span><span>EPE Transição</span></button>
        <button class="tab-btn" data-tab="cnpe" onclick="ativarAba('cnpe')"><span class="nav-icon">▣</span><span>CNPE</span></button>
        <button class="tab-btn" data-tab="cmse" onclick="ativarAba('cmse')"><span class="nav-icon">◫</span><span>CMSE</span></button>
        <button class="tab-btn" data-tab="megawhat" onclick="ativarAba('megawhat')"><span class="nav-icon">▶</span><span>Minuto MegaWhat</span></button>
        <div class="nav-section-label">Recursos</div>
        <button class="tab-btn" data-tab="ons" onclick="ativarAba('ons'); document.querySelector('#grid-ons')?.scrollIntoView({{behavior:'smooth'}})"><span class="nav-icon">♧</span><span>Indicadores</span></button>
        <button class="tab-btn" data-tab="ons" onclick="ativarAba('ons'); document.querySelector('#grid-ons')?.scrollIntoView({{behavior:'smooth'}})"><span class="nav-icon">⇩</span><span>Exportações</span></button>
    </nav>
    <div class="sidebar-footer"><span class="live-dot"></span>Atualização automática<br><span style="opacity:.75;">Período: {PERIODO_ATUAL}</span></div>
</aside>
<button id="sidebar-reopen" class="sidebar-reopen" type="button" onclick="alternarMenuLateral(false)" aria-label="Reabrir menu" title="Reabrir menu">›</button>
<main class="main-content">
    <header class="topbar"><div><p class="eyebrow">CENTRO DE OPERAÇÕES E INTELIGÊNCIA</p><h1>Visão Geral do Sistema Elétrico</h1><p class="top-subtitle">Monitoramento integrado de dados oficiais do setor elétrico brasileiro.</p></div><div class="topbar-actions"><span class="period-badge">Período: {PERIODO_ATUAL}</span><span class="updated-badge">Atualizado agora</span><button class="header-export" type="button" onclick="ativarAba('ons')">Exportar dados</button><button class="btn-theme" onclick="toggleTheme()" id="btn-theme">Tema escuro</button></div></header>
    <section class="filter-bar"><div class="filter-item"><span>Janela de análise</span><strong>Últimos {periodo_dias} dias</strong></div><div class="filter-item"><span>Fontes conectadas</span><strong>ONS · CCEE · ANEEL</strong></div><div class="filter-item"><span>Status do ambiente</span><strong style="color:#16845b;">● Operacional</strong></div><div class="filter-note">Use os filtros de período dentro de cada card para aprofundar a análise sem perder o contexto executivo.</div></section>

    <section id="painel-visao-geral" class="tab-panel active">
        <div class="section-heading"><div><h2>Resumo executivo</h2><p>Principais sinais do sistema a partir dos dados carregados.</p></div><span class="section-status">Dados processados automaticamente</span></div>
        <div class="kpi-grid">
            <div class="kpi-tile"><span class="kpi-accent"></span><span class="kpi-label">CVU consolidado</span><strong id="overview-kpi-cvu" class="kpi-value">—</strong><span class="kpi-foot">Usinas térmicas</span></div>
            <div class="kpi-tile"><span class="kpi-accent"></span><span class="kpi-label">PLD de referência</span><strong id="overview-kpi-pld" class="kpi-value">—</strong><span class="kpi-foot">CCEE · submercados</span></div>
            <div class="kpi-tile"><span class="kpi-accent"></span><span class="kpi-label">Carga monitorada</span><strong id="overview-kpi-carga" class="kpi-value">—</strong><span class="kpi-foot">ONS · série recente</span></div>
            <div class="kpi-tile"><span class="kpi-accent"></span><span class="kpi-label">Armazenamento</span><strong id="overview-kpi-ear" class="kpi-value">—</strong><span class="kpi-foot">Energia armazenada</span></div>
            <div class="kpi-tile"><span class="kpi-accent"></span><span class="kpi-label">Indicadores carregados</span><strong class="kpi-value">{len(dados_coletados)}</strong><span class="kpi-foot">Processamento concluído</span></div>
            <div class="kpi-tile"><span class="kpi-accent"></span><span class="kpi-label">Alertas identificados</span><strong class="kpi-value">{alertas_count}</strong><span class="kpi-foot">Requerem acompanhamento</span></div>
        </div>
        <div class="overview-feature">
            <article class="overview-cvu-card"><div class="overview-card-header"><div><h2>Empilhamento térmico do SIN</h2><p>POR CVU (R$/MWh)</p></div><span class="overview-chart-note">Colunas ordenadas por usina<br>Valor mais recente disponível</span></div><div id="overview-cvu-summary"></div><div id="overview-cvu-chart" class="overview-cvu-chart"></div></article>
            <aside class="overview-detail"><h3>Detalhes do indicador</h3><div class="detail-item"><span>Usinas exibidas</span><strong id="overview-detail-usinas">—</strong></div><div class="detail-item"><span>Último valor de referência</span><strong id="overview-detail-valor">—</strong></div><div class="detail-item"><span>Fonte de dados</span><strong>ONS</strong></div><a id="overview-detail-source" class="detail-source" href="#" target="_blank" rel="noopener">Abrir fonte oficial ↗</a><button class="detail-button" type="button" onclick="ativarAba('ons')">Ver análise completa</button></aside>
        </div>
        <div class="overview-lower-grid"><article class="overview-lower-card"><h3>Dados ONS</h3><p>Indicadores operacionais, carga, geração, hidrologia, armazenamento e transmissão.</p></article><article class="overview-lower-card"><h3>Dados CCEE</h3><p>PLD e informações de mercado organizadas por submercado e período.</p></article><article class="overview-lower-card"><h3>Dados ANEEL</h3><p>Links para painéis, bases abertas e referências regulatórias oficiais.</p></article></div>
    </section>

    <section id="painel-ons" class="tab-panel"><div class="section-heading"><div><h2>Dados ONS</h2><p>Indicadores operacionais e séries históricas do Operador Nacional do Sistema.</p></div><span class="section-status">Fonte oficial ONS</span></div><div id="grid-ons" class="grid"></div></section>
    <section id="painel-ccee" class="tab-panel"><div class="section-heading"><div><h2>Dados CCEE</h2><p>Mercado, PLD e referências públicas da Câmara de Comercialização.</p></div><span class="section-status">Fonte oficial CCEE</span></div>{ccee_modules_html}<div id="grid-ccee" class="grid" style="margin-top:20px;"></div></section>
    <section id="painel-aneel" class="tab-panel"><div class="section-heading"><div><h2>Dados ANEEL</h2><p>Painéis, bases abertas e indicadores regulatórios.</p></div><span class="section-status">Fonte oficial ANEEL</span></div>{aneel_modules_html}</section>

    <section id="painel-meteorologia" class="tab-panel">
        <div class="section-heading"><div><h2>Mapa meteorológico</h2><p>Condições atmosféricas e previsão para o Sistema Interligado Nacional.</p></div><span class="section-status">Fonte oficial Windy</span></div>
        <div class="weather-toolbar">
            <label class="weather-control">Camada
                <select id="windy-layer" onchange="alterarCamadaWindy(this.value)">
                    <option value="wind">Vento</option><option value="rain">Chuva e neve</option><option value="temp">Temperatura</option><option value="clouds">Nuvens</option><option value="satellite">Satélite</option>
                </select>
            </label>
            <label class="weather-control">Modelo
                <select id="windy-model" onchange="alterarModeloWindy(this.value)"><option value="ecmwf">ECMWF</option><option value="gfs">GFS</option><option value="icon">ICON</option></select>
            </label>
            <span class="weather-context">Brasil · superfície · mapa interativo</span>
            <a class="weather-open-link" href="https://www.windy.com/" target="_blank" rel="noopener">Abrir Windy ↗</a>
        </div>
        <div class="weather-layout">
            <div class="weather-map-card"><iframe id="windy-map" title="Mapa meteorológico Windy" loading="lazy" allowfullscreen></iframe></div>
            <aside class="weather-info-card"><div class="weather-info-heading"><h3>Camadas meteorológicas</h3><span class="weather-live-dot"></span></div><p>Use os controles do mapa para alternar entre vento, chuva, temperatura e nuvens.</p><div class="weather-info-row"><span>Modelo selecionado</span><strong id="windy-model-label">ECMWF</strong></div><div class="weather-info-row"><span>Área de referência</span><strong>Brasil e entorno do SIN</strong></div><div class="weather-info-row"><span>Fonte</span><strong>Windy.com</strong></div><a class="weather-info-source" href="https://embed.windy.com/" target="_blank" rel="noopener">Configuração oficial do mapa ↗</a></aside>
        </div>
        <div class="weather-bottom-grid"><article class="weather-bottom-card"><h3>Previsão por região</h3><p>Explore o mapa e aproxime as regiões Norte, Nordeste, Centro-Oeste, Sudeste e Sul para analisar as condições locais.</p></article><article class="weather-bottom-card"><h3>Uso no monitoramento</h3><p>O mapa funciona como uma camada complementar aos indicadores ONS, CCEE e ANEEL, sem alterar a coleta dos dados do painel.</p></article><article class="weather-bottom-card"><h3>Atualização</h3><p>As camadas são carregadas diretamente do Windy e requerem conexão com a internet para atualização e interação.</p></article></div>
    </section>
    <section id="painel-epe-elec" class="tab-panel">{epe_elec_html}</section>
    <section id="painel-epe-plan" class="tab-panel">{epe_plan_html}</section>
    <section id="painel-epe-oil" class="tab-panel">{epe_oil_html}</section>
    <section id="painel-epe-trans" class="tab-panel">{epe_transition_html}</section>
    <section id="painel-cnpe" class="tab-panel">{cnpe_html}</section>
    <section id="painel-cmse" class="tab-panel">{cmse_html}</section>
    <section id="painel-megawhat" class="tab-panel">{megawhat_html}</section>
    <section id="painel-denergia" class="tab-panel">
        <div class="section-heading"><div><h2>Denergia</h2><p>Índices de mercado, curva forward, modulação e PLD.</p></div><span class="section-status">Portal Denergia</span></div>
        <div class="external-portal-toolbar"><span class="external-portal-status">Conteúdo externo carregado</span><span class="external-portal-note">O portal é exibido dentro do padrão do painel e continua sendo atualizado diretamente pela fonte original.</span><a class="external-portal-link" href="https://denergia.com.br/login" target="_blank" rel="noopener">Abrir Denergia ↗</a></div>
        <div class="external-portal-layout"><div class="external-portal-frame-card"><iframe class="external-portal-frame" title="Dashboard Denergia" src="https://denergia.com.br/login" loading="lazy" allowfullscreen></iframe></div><aside class="external-portal-info"><h3>Resumo do portal</h3><p>Área independente para consultar os indicadores públicos apresentados pelo Denergia.</p><div class="external-portal-info-row"><span>Conteúdos</span><strong>Curva Forward, Modulação e PLD</strong></div><div class="external-portal-info-row"><span>Atualização</span><strong>Direta no portal original</strong></div><div class="external-portal-info-row"><span>Fonte</span><strong>Denergia / Dcide</strong></div><a class="external-portal-link" href="https://denergia.com.br/login" target="_blank" rel="noopener">Abrir em nova aba</a><div class="external-portal-warning">Se o navegador bloquear a incorporação, use o botão acima para abrir o portal diretamente. Nenhum dado interno do seu painel depende deste módulo.</div></aside></div>
    </section>
    <section id="painel-cammesa" class="tab-panel">
        <div class="section-heading"><div><h2>CAMMESA</h2><p>Operação, programação, renováveis, relatórios, MATER e documentos do sistema argentino.</p></div><span class="section-status">Portal oficial CAMMESA</span></div>
        <div class="external-portal-toolbar"><span class="external-portal-status">Conteúdo externo carregado</span><span class="external-portal-note">O portal CAMMESA é mantido em um módulo isolado para não interferir nos dados, gráficos e downloads do projeto.</span><a class="external-portal-link" href="https://cammesaweb.cammesa.com/" target="_blank" rel="noopener">Abrir CAMMESA ↗</a></div>
        <div class="external-portal-layout"><div class="external-portal-frame-card"><iframe class="external-portal-frame" title="Portal CAMMESA" src="https://cammesaweb.cammesa.com/" loading="lazy" allowfullscreen></iframe></div><aside class="external-portal-info"><h3>Resumo do portal</h3><p>Área independente para acessar o conteúdo público do portal oficial CAMMESA sem alterar a lógica do dashboard.</p><div class="external-portal-info-row"><span>Conteúdos</span><strong>Operação, relatórios e renováveis</strong></div><div class="external-portal-info-row"><span>Recursos</span><strong>Programação, MATER e documentos</strong></div><div class="external-portal-info-row"><span>Fonte</span><strong>CAMMESA</strong></div><a class="external-portal-link" href="https://cammesaweb.cammesa.com/" target="_blank" rel="noopener">Abrir em nova aba</a><div class="external-portal-warning">Se o navegador bloquear a incorporação, use o botão acima para abrir o portal diretamente. Os cards ONS, CCEE, ANEEL e Windy continuam independentes.</div></aside></div>
    </section>
    <div id="painel-detalhe"><button class="detail-back" onclick="voltarDashboard()">← Voltar para o painel</button><h2 id="detalhe-titulo" style="color:var(--text-main); margin:0 0 15px;"></h2><div class="detail-layout"><div id="detalhe-grafico"></div><div id="detalhe-info"></div></div></div>
    <div id="grid-dashboard" class="grid" style="display:none">{''.join(cards_html) if isinstance(cards_html, list) else cards_html_string}</div>
    <footer class="footer">Dados oficiais · Powered by Dados Abertos ONS · CCEE · ANEEL · Gerado em {datetime.now().strftime('%d/%m/%Y %H:%M')} · <a href="https://dados.ons.org.br/" target="_blank" rel="noopener">Fontes oficiais ↗</a></footer>
</main></div></div>
<script>
    // ── Login local funcional e editável no notebook ─────────────────────────
    const LOGIN_USUARIO_CONFIG = {LOGIN_USUARIO_JS};
    const LOGIN_SENHA_CONFIG = {LOGIN_SENHA_JS};
    function realizarLogin(event) {{
        event.preventDefault();
        const u = document.getElementById('login-user').value;
        const p = document.getElementById('login-pass').value;
        const erro = document.getElementById('login-error');
        if (u === LOGIN_USUARIO_CONFIG && p === LOGIN_SENHA_CONFIG) {{
            sessionStorage.setItem('dashboard_authenticated','1');
            document.getElementById('login-screen').style.display='none';
            document.getElementById('app-shell').style.display='block';
            return false;
        }}
        erro.textContent='Usuário ou senha incorretos.';
        document.getElementById('login-pass').value='';
        return false;
    }}
    (function() {{
        if (sessionStorage.getItem('dashboard_authenticated') === '1') {{
            document.getElementById('login-screen').style.display='none';
            document.getElementById('app-shell').style.display='block';
        }}
    }})();

    function alternarMenuLateral(forcarAberto) {{
        var recolhido = typeof forcarAberto === 'boolean' ? !forcarAberto : !document.body.classList.contains('sidebar-collapsed');
        document.body.classList.toggle('sidebar-collapsed', recolhido);
        var toggle = document.getElementById('sidebar-toggle');
        var reopen = document.getElementById('sidebar-reopen');
        if (toggle) {{
            toggle.setAttribute('aria-expanded', String(!recolhido));
            toggle.setAttribute('aria-label', recolhido ? 'Reabrir menu' : 'Ocultar menu');
            var label = toggle.querySelector('.sidebar-toggle-label');
            if (label) label.textContent = recolhido ? 'Reabrir menu' : 'Ocultar menu';
        }}
        if (reopen) reopen.setAttribute('aria-hidden', String(!recolhido));
        localStorage.setItem('dashboard_sidebar_collapsed', recolhido ? '1' : '0');
    }}
    (function() {{
        if (localStorage.getItem('dashboard_sidebar_collapsed') === '1') alternarMenuLateral();
    }})();

    // ── Organização das abas ONS/CCEE/ANEEL ────────────────────────────────────
    function ativarAba(tab) {{
        document.querySelectorAll('.tab-panel').forEach(p => p.classList.remove('active'));
        document.querySelectorAll('.tab-btn').forEach(b => b.classList.toggle('active', b.dataset.tab === tab));
        document.getElementById('painel-' + tab).classList.add('active');
        localStorage.setItem('dashboard_tab', tab);
        if (tab === 'visao-geral') setTimeout(montarVisaoGeral, 0);
    }}
    function organizarAbas() {{
        var origem = document.getElementById('grid-dashboard');
        var ons = document.getElementById('grid-ons');
        var ccee = document.getElementById('grid-ccee');
        Array.from(origem.querySelectorAll('.card-item')).forEach(card => {{
            var texto = card.querySelector('strong')?.textContent || '';
        var destino = (texto.indexOf('PLD ') >= 0) ? ccee : ons;
            destino.appendChild(card);
        }});
    }}
    organizarAbas();
    var abaSalva = localStorage.getItem('dashboard_tab');
    if (['visao-geral', 'meteorologia', 'denergia', 'cammesa', 'ons', 'ccee', 'aneel', 'epe-elec', 'epe-plan', 'epe-oil', 'epe-trans', 'cnpe', 'cmse', 'megawhat'].includes(abaSalva)) ativarAba(abaSalva); else ativarAba('visao-geral');

    // ── Renderiza gráficos iniciais ──────────────────────────────────────────
    document.querySelectorAll('.plotly-lazy').forEach(el => {{
        try {{
            var spec = JSON.parse(el.dataset.spec);
            Plotly.newPlot(el, spec.data, spec.layout, {{responsive: true, displaylogo: false}});
        }} catch(e) {{
            console.error('Falha ao renderizar gráfico:', e);
            el.innerHTML = '<p style="color:var(--text-muted)">Dados indisponíveis para visualização.</p>';
        }}
    }});

    // ── Popula date pickers com o período atual ──────────────────────────────
    (function() {{
        var hoje = new Date();
        var fim  = hoje.toISOString().substring(0, 10);

        var periodoDias = {periodo_dias};  // vem do Python
        var inicio = new Date(hoje);
        inicio.setDate(inicio.getDate() - periodoDias);
        var inicioStr = inicio.toISOString().substring(0, 10);

        document.querySelectorAll('.card-item').forEach(card => {{
            var di = card.querySelector('.date-inicio');
            var df = card.querySelector('.date-fim');
            if (di) di.value = inicioStr;
            if (df) df.value = fim;
        }});
    }})();

    // ── Filtro de data por card ──────────────────────────────────────────────
    document.querySelectorAll('.card-item').forEach(card => {{
        var btnAtualizar = card.querySelector('.btn-atualizar');
        if (!btnAtualizar) return;

        btnAtualizar.addEventListener('click', function() {{
            var slug     = card.dataset.slug;
            var inicio   = card.querySelector('.date-inicio').value;
            var fim      = card.querySelector('.date-fim').value;
            var msg      = card.querySelector('.msg-status');
            var plotDiv  = card.querySelector('.plotly-lazy');

            if (!inicio || !fim) {{
                msg.textContent = '⚠ Preencha as duas datas.';
                msg.style.color = '#E63946';
                return;
            }}
            if (inicio > fim) {{
                msg.textContent = '⚠ Data início maior que fim.';
                msg.style.color = '#E63946';
                return;
            }}

            msg.textContent = '⏳ Buscando...';
            msg.style.color = '#888';
            btnAtualizar.disabled = true;

            // Calcula periodo_dias a partir do range escolhido
            var ms   = new Date(fim) - new Date(inicio);
            var dias = Math.ceil(ms / 86400000) + 1;

            var embedded = card.dataset.csvB64 || '';
            if (!embedded) {{
                msg.textContent = '❌ Dados locais indisponíveis para este card';
                msg.style.color = '#E63946';
                btnAtualizar.disabled = false;
                return;
            }}
            // O filtro usa exclusivamente o CSV completo embutido pelo backend.
            // Isso evita CORS e elimina qualquer fetch no navegador.
            Promise.resolve().then(() => {{
                var binary = atob(embedded);
                var bytes = Uint8Array.from(binary, ch => ch.charCodeAt(0));
                return new TextDecoder('utf-8').decode(bytes).replace(/^\uFEFF/, '');
            }})
                .then(csvText => {{
                    // Parse CSV simples; o recurso selecionado é convertido para texto antes do filtro.
                    var linhas = csvText.replace(/^\uFEFF/, '').split(/\\r?\\n/).filter(l => l.trim().length);
                    if (!linhas.length) throw new Error('Recurso vazio');
                    var header = parseCSVLine(linhas[0]).map(h => h.replace(/"/g,'').trim());

                    function normalizarDataONS(valor) {{
                        var texto = String(valor ?? '').replace(/^\"|\"$/g, '').trim();
                        var m = texto.match(/^(\d{{4}})[\/-](\d{{1,2}})[\/-](\d{{1,2}})/);
                        if (m) return m[1] + '-' + String(m[2]).padStart(2, '0') + '-' + String(m[3]).padStart(2, '0');
                        m = texto.match(/^(\d{{1,2}})[\/-](\d{{1,2}})[\/-](\d{{4}})/);
                        if (m) return m[3] + '-' + String(m[2]).padStart(2, '0') + '-' + String(m[1]).padStart(2, '0');
                        m = texto.match(/^(\d{{4}})(\d{{2}})(\d{{2}})/);
                        if (m) return m[1] + '-' + m[2] + '-' + m[3];
                        return '';
                    }}
                    function parseCSVLine(line) {{
                        var result = [], cur = '', inQ = false;
                        for (var i = 0; i < line.length; i++) {{
                            var c = line[i];
                            if (c === '"') {{ inQ = !inQ; }}
                            else if ((c === ',' || c === ';') && !inQ) {{ result.push(cur.trim()); cur = ''; }}
                            else {{ cur += c; }}
                        }}
                        result.push(cur.trim());
                        return result;
                    }}
                    var rows = linhas.slice(1).map(l => parseCSVLine(l));
                    // Os datasets ONS usam nomes variados: ear_data, dat_iniciosemana,
                    // din_instante, entre outros. Se o nome não for conhecido, detecta
                    // a primeira coluna com valores reconhecíveis como data.
                    var candidatasData = ['din_instante','dat_referencia','data_referencia','momento','data','ear_data','dat_iniciosemana','dat_fimsemana','din_medicao','dat_medicao','timestamp','dt_'];
                    var tIdx = header.findIndex(h => candidatasData.some(c => h.toLowerCase().includes(c)));
                    if (tIdx < 0) tIdx = header.findIndex((h, i) => rows.slice(0, 30).some(r => normalizarDataONS(r[i])));
                    if (tIdx < 0) throw new Error('Coluna de data não identificada');
                    var tcol = header[tIdx];

                    var numCols = header.filter((h, i) => i !== tIdx);
                    var vcolHint = card.dataset.vcol || ''; var vcol = (vcolHint && header.includes(vcolHint)) ? vcolHint : numCols.find(h => ['val_','valor','geracao','carga','ear','ena','cmo','pld'].some(c => h.toLowerCase().includes(c))) || numCols[0];
                    var vIdx = header.indexOf(vcol);

                    // Filtra pelo range de datas escolhido, preservando todas as séries/grupos.
                    var filtrado = rows.filter(r => {{
                        var d = normalizarDataONS(r[tIdx]);
                        return d && d >= inicio && d <= fim;
                    }});
                    if (!filtrado.length) throw new Error('Sem dados no período selecionado');

                    var groupCandidates = ['nom_subsistema','nom_ree','nom_bacia','nom_usina','nom_reservatorio','nom_reservatório','submercado','nom_tipo'];
                    var isCvuColumns = card.dataset.chartMode === 'cvu-consolidado';
                    var isIntercambio = card.dataset.chartMode === 'intercambio-direcional';
                    var regiaoIntercambio = nomeIndicador.toLowerCase().includes('nordeste') ? 'NORDESTE' : 'SUL';
                    var gIdx = (card.dataset.chartMode === 'cvu-combo' || isCvuColumns)
                        ? header.findIndex(h => h.toLowerCase().trim() === 'nom_usina')
                        : header.findIndex(h => groupCandidates.includes(h.toLowerCase().trim()));
                    var usinaSelect = card.querySelector('.usina-select');
                    var usinaEscolhida = usinaSelect ? usinaSelect.value : '__TODAS__';
                    if (usinaEscolhida !== '__TODAS__' && gIdx >= 0) {{
                        filtrado = filtrado.filter(r => ((r[gIdx] || '').replace(/"/g,'').trim()) === usinaEscolhida);
                    }}
                    if (isIntercambio) {{
                        var origemIdxInter = header.findIndex(h => ['nom_subsistema_origem','subsistema_origem'].includes(h.toLowerCase().trim()));
                        var destinoIdxInter = header.findIndex(h => ['nom_subsistema_destino','subsistema_destino'].includes(h.toLowerCase().trim()));
                        if (origemIdxInter < 0 || destinoIdxInter < 0) throw new Error('Colunas de origem/destino não identificadas');
                        filtrado = filtrado.filter(r => {{
                            var origem = (r[origemIdxInter] || '').replace(/"/g,'').trim().toUpperCase();
                            var destino = (r[destinoIdxInter] || '').replace(/"/g,'').trim().toUpperCase();
                            return (origem === 'SUDESTE' && destino === regiaoIntercambio) || (origem === regiaoIntercambio && destino === 'SUDESTE');
                        }});
                        if (!filtrado.length) throw new Error('Sem intercâmbios da região no período selecionado');
                    }}
                    var specAtual = {{}};
                    try {{ specAtual = JSON.parse(plotDiv.dataset.spec || '{{}}'); }} catch (_) {{ specAtual = {{}}; }}
                    var cores = (specAtual.data || []).map(t => (t.marker && t.marker.color) || (t.line && t.line.color)).filter(Boolean);
                    if (!cores.length) cores = document.body.classList.contains('dark') ? ['#22D3EE','#4ADE80','#FBBF24','#C084FC'] : ['#A96E5F','#9E6A5C','#B17A6B','#936052'];
                    var nomeIndicador = card.querySelector('strong')?.textContent || 'Indicador';
                    var unidadeCard = card.dataset.unit || '';
                    var traces;

                    if (isIntercambio) {{
                        var recebidosInter = new Map(), enviadosInter = new Map();
                        filtrado.forEach(r => {{
                            var dataInter = normalizarDataONS(r[tIdx]);
                            var origemInter = (r[origemIdxInter] || '').replace(/"/g,'').trim().toUpperCase();
                            var destinoInter = (r[destinoIdxInter] || '').replace(/"/g,'').trim().toUpperCase();
                            var valorInter = parseFloat((r[vIdx] || '').replace(/"/g,'').replace(',','.'));
                            if (!dataInter || isNaN(valorInter)) return;
                            var mapaInter = (origemInter === 'SUDESTE' && destinoInter === regiaoIntercambio) ? recebidosInter : enviadosInter;
                            mapaInter.set(dataInter, (mapaInter.get(dataInter) || 0) + Math.abs(valorInter));
                        }});
                        var nomeRegiaoInter = regiaoIntercambio === 'NORDESTE' ? 'Nordeste' : 'Sul';
                        traces = [
                            {{x: Array.from(recebidosInter.keys()), y: Array.from(recebidosInter.values()), type: 'bar', name: nomeRegiaoInter + ' recebendo do Sudeste', marker: {{color: '#4472C4'}}, hovertemplate: nomeRegiaoInter + ' recebendo do Sudeste<br>%{{x|%d/%m/%Y}}<br>%{{y:,.0f}} ' + (unidadeCard || 'MWmed') + '<extra></extra>'}},
                            {{x: Array.from(enviadosInter.keys()), y: Array.from(enviadosInter.values()).map(v => -Math.abs(v)), type: 'bar', name: nomeRegiaoInter + ' enviado para Sudeste', marker: {{color: '#ED7D31'}}, hovertemplate: nomeRegiaoInter + ' enviado para Sudeste<br>%{{x|%d/%m/%Y}}<br>%{{y:,.0f}} ' + (unidadeCard || 'MWmed') + '<extra></extra>'}}
                        ];
                    }} else if (isCvuColumns) {{
                        // O card de CVU usa uma coluna por usina, sempre com o último valor do recorte.
                        var cvuPorUsina = new Map();
                        filtrado.forEach(r => {{
                            var usina = gIdx >= 0 ? (r[gIdx] || '').replace(/"/g,'').trim() : '';
                            var data = normalizarDataONS(r[tIdx]);
                            var valor = parseFloat((r[vIdx] || '').replace(/"/g,'').replace(',','.'));
                            if (!usina || isNaN(valor)) return;
                            var anterior = cvuPorUsina.get(usina);
                            if (!anterior || data >= anterior.data) {{
                                var regIdx = header.findIndex(h => h.toLowerCase().trim() === 'nom_subsistema');
                                var regiao = regIdx >= 0 ? (r[regIdx] || '').replace(/"/g,'').trim() : nomeIndicador;
                                cvuPorUsina.set(usina, {{data: data, valor: valor, regiao: regiao}});
                            }}
                        }});
                        var barras = Array.from(cvuPorUsina.entries())
                            .map(([usina, item]) => ({{usina: usina, data: item.data, valor: item.valor, regiao: item.regiao}}))
                            .sort((a, b) => a.valor - b.valor);
                        var formatarDataCvu = function(valor) {{
                            var texto = String(valor || '');
                            var partes = texto.substring(0, 10).split('-');
                            return partes.length === 3 && partes[0].length === 4
                                ? partes[2] + '/' + partes[1] + '/' + partes[0]
                                : texto.substring(0, 10);
                        }};
                        var formatarNumeroCvu = function(valor) {{
                            return new Intl.NumberFormat('pt-BR', {{minimumFractionDigits: 2, maximumFractionDigits: 2}}).format(valor);
                        }};
                        traces = [{{
                            x: barras.map(item => item.usina),
                            y: barras.map(item => item.valor),
                            type: 'bar',
                            name: 'CVU',
                            marker: {{color: barras.map(item => {{
                                var paleta = {{'Norte':'#0072B2','Sul':'#009E73','Sudeste/Centro-Oeste':'#D55E00','Nordeste':'#CC79A7'}};
                                return paleta[item.regiao] || '#A96E5F';
                            }}), line: {{color: 'rgba(100,60,50,0.30)', width: 0.4}}}},
                            customdata: barras.map(item => [item.regiao, formatarDataCvu(item.data), formatarNumeroCvu(item.valor)]),
                            hovertemplate: '<b>%{{x}}</b><br>Região: %{{customdata[0]}}<br>Data: %{{customdata[1]}}<br>CVU: %{{customdata[2]}} ' + (unidadeCard || 'R$/MWh') + '<extra></extra>'
                        }}];
                    }} else {{
                        var groupValues = gIdx >= 0 ? [...new Set(filtrado.map(r => (r[gIdx] || '').replace(/"/g,'').trim()).filter(Boolean))] : [];
                        var serieGrupos = groupValues.length > 1 ? groupValues : [null];
                        traces = serieGrupos.map((grupo, serieIdx) => {{
                            var subset = grupo === null ? filtrado : filtrado.filter(r => ((r[gIdx] || '').replace(/"/g,'').trim()) === grupo);
                            var xsSerie = subset.map(r => normalizarDataONS(r[tIdx])).filter(Boolean);
                            var ysSerie = subset.map(r => parseFloat((r[vIdx] || '').replace(/"/g,'').replace(',','.')));
                            return {{
                                x: xsSerie, y: ysSerie, type: 'scatter', mode: 'lines', name: grupo || nomeIndicador,
                                connectgaps: false, fill: serieIdx === 0 && serieGrupos.length === 1 ? 'tozeroy' : 'none',
                                line: {{color: cores[serieIdx % cores.length], width: 2}},
                                hovertemplate: '<b>' + (grupo || nomeIndicador) + '</b><br>Região: ' + nomeIndicador.replace('🟢 ','').replace('⚠ ','') + '<br>Data: %{{x|%d/%m/%Y}}<br>Valor: %{{y:,.2f}} ' + (unidadeCard || '') + '<extra></extra>'
                            }};
                        }}).filter(t => t.x.length);
                    }}

                    Plotly.react(plotDiv, traces, {{
                        template: getPlotlyTemplate(),
                        paper_bgcolor: document.body.classList.contains('dark') ? '#221952' : '#ffffff',
                        plot_bgcolor:  document.body.classList.contains('dark') ? '#221952' : '#ffffff',
                        font: {{color: document.body.classList.contains('dark') ? '#F4F7FF' : '#2d2d3a'}},
                        hovermode: isCvuColumns ? 'closest' : 'x unified', showlegend: isCvuColumns ? false : traces.length > 1,
                        barmode: isIntercambio ? 'relative' : undefined,
                        legend: card.dataset.chartMode === 'cvu-combo' ? {{orientation: 'v', x: 1.02, xanchor: 'left', y: 1, font: {{size: 9}}}} : undefined,
                        hoverlabel: {{bgcolor: document.body.classList.contains('dark') ? '#111827' : '#ffffff', font: {{color: document.body.classList.contains('dark') ? '#F4F7FF' : '#1f2937', size: 12}}}},
                        height: isCvuColumns ? 390 : (isIntercambio ? 470 : 250),
                        bargap: isCvuColumns ? 0.16 : undefined,
                        xaxis: isCvuColumns ? {{title: '', showgrid: false, showline: true, linecolor: '#d9d9d9', tickangle: -90, tickfont: {{size: 8, color: '#6b7280'}}, automargin: true}} : undefined,
                        yaxis: isCvuColumns ? {{title: unidadeCard || 'R$/MWh', showgrid: true, gridcolor: '#dedede', zeroline: false, rangemode: 'tozero', automargin: true}} : (unidadeCard ? {{title: unidadeCard, automargin: true}} : undefined),
                        margin: isCvuColumns ? {{l:58, r:18, t:16, b:112}} : (isIntercambio ? {{l:58, r:145, t:16, b:48}} : (card.dataset.chartMode === 'cvu-combo' ? {{l:10, r:145, t:10, b:30}} : {{l:10, r:10, t:10, b:10}}))
                    }}, {{responsive: true, displaylogo: false}});

                    msg.textContent = '✓ ' + filtrado.length + ' registros';
                    msg.style.color = '#2A9D8F';
                }})
                .catch(err => {{
                    msg.textContent = '❌ ' + err.message;
                    msg.style.color = '#E63946';
                }})
                .finally(() => {{ btnAtualizar.disabled = false; }});
        }});
    }});

    // ── Toggle de tema ───────────────────────────────────────────────────────
    function getPlotlyTemplate() {{
        return document.body.classList.contains('dark') ? 'plotly_dark' : 'plotly_white';
    }}
    function updateChartsTheme() {{
        var isDark = document.body.classList.contains('dark');
        var paperColor = isDark ? '#221952' : '#ffffff';
        var plotColor  = isDark ? '#221952' : '#ffffff';
        var fontColor  = isDark ? '#e0e6ff' : '#333333';
        var gridColor  = isDark ? '#3d2f7a' : '#eeeeee';
        document.querySelectorAll('.plotly-lazy').forEach(el => {{
            if (!el._fullLayout) return;
            Plotly.relayout(el, {{
                'paper_bgcolor': paperColor,
                'plot_bgcolor':  plotColor,
                'font.color':    fontColor,
                'xaxis.gridcolor': gridColor,
                'yaxis.gridcolor': gridColor,
                'xaxis.linecolor': gridColor,
                'yaxis.linecolor': gridColor,
                'hoverlabel.bgcolor': isDark ? '#111827' : '#ffffff',
                'hoverlabel.font.color': isDark ? '#F4F7FF' : '#1f2937',
            }});
        }});
    }}
    function toggleTheme() {{
        var isDark = document.body.classList.toggle('dark');
        var btn = document.getElementById('btn-theme');
        btn.textContent = isDark ? '☀️ Tema Claro' : '🌙 Tema Escuro';
        localStorage.setItem('theme', isDark ? 'dark' : 'light');
        updateChartsTheme();
    }}
    (function() {{
        var saved = localStorage.getItem('theme');
        if (saved === 'dark') {{
            document.body.classList.add('dark');
            document.getElementById('btn-theme').textContent = '☀️ Tema Claro';
            updateChartsTheme();
        }}
    }})();

    // ── Detalhe do card ──────────────────────────────────────────────────────
    function abrirDetalhe(el) {{
        var card    = el.closest('.card-item');
        var titulo  = el.textContent.trim();
        var specEl  = card.querySelector('.plotly-lazy');

        document.querySelectorAll('.tab-panel').forEach(p => p.classList.remove('active'));
        document.getElementById('painel-detalhe').style.display = 'block';
        document.getElementById('detalhe-titulo').textContent   = titulo;

        var grafDiv = document.getElementById('detalhe-grafico');
        var infoDiv = document.getElementById('detalhe-info');

        grafDiv.innerHTML = '<p style="color:var(--text-muted)">⏳ Carregando...</p>';
        infoDiv.innerHTML = '<p style="color:var(--text-muted)">⏳ Calculando...</p>';

        try {{
            var spec  = JSON.parse(specEl.dataset.spec);
            var isDark = document.body.classList.contains('dark');

            // Ajusta layout para tamanho maior
            var layout = Object.assign({{}}, spec.layout, {{
                height: 420,
                margin: {{l:50, r:20, t:20, b:50}},
                paper_bgcolor: isDark ? '#221952' : '#ffffff',
                plot_bgcolor:  isDark ? '#221952' : '#ffffff',
                font: {{color: isDark ? '#F4F7FF' : '#2d2d3a'}},
                hovermode: 'x unified',
                hoverlabel: {{bgcolor: isDark ? '#111827' : '#ffffff', font: {{color: isDark ? '#F4F7FF' : '#1f2937', size: 12}}}}
            }});

            Plotly.newPlot(grafDiv, spec.data, layout, {{responsive: true, displaylogo: false}});

            // Extrai xs e ys do primeiro trace para estatísticas
            var trace = spec.data[0] || {{}};
            var xs = trace.x || [];
            var ys = (trace.y || []).map(v => parseFloat(v)).filter(v => !isNaN(v));

            if (!ys.length) throw new Error('Sem dados numéricos no gráfico');

            var atual    = ys[ys.length - 1];
            var anterior = ys[ys.length - 2];
            var variacao = (anterior && anterior !== 0) ? ((atual - anterior) / Math.abs(anterior) * 100) : null;
            var minVal   = Math.min(...ys);
            var maxVal   = Math.max(...ys);
            var media    = ys.reduce((a,b) => a+b, 0) / ys.length;
            var fmt      = v => v >= 1e6 ? (v/1e6).toFixed(2)+'M' : v >= 1e3 ? (v/1e3).toFixed(1)+'k' : v.toFixed(2);
            var varSeta  = variacao === null ? '' : (variacao >= 0 ? '▲' : '▼');
            var varCor   = variacao === null ? 'var(--text-muted)' : (variacao >= 0 ? '#2A9D8F' : '#E63946');

            // Últimos 10 pares data/valor
            var ultimos = xs.slice(-10).map((d,i) => {{
                var yi = ys.slice(-10)[i];
                return `<tr style="border-bottom:1px solid var(--border)">
                    <td style="padding:6px 8px; color:var(--text-muted); font-size:12px;">${{String(d).substring(0,10)}}</td>
                    <td style="padding:6px 8px; font-weight:600; font-size:12px;">${{fmt(yi)}}</td>
                </tr>`;
            }}).reverse().join('');

            infoDiv.innerHTML = `
                <div style="margin-bottom:20px;">
                    <div style="font-size:12px; color:var(--text-muted); margin-bottom:4px;">Valor atual</div>
                    <div style="font-size:32px; font-weight:700; color:var(--accent);">${{fmt(atual)}}</div>
                    ${{variacao !== null ? `<div style="font-size:14px; color:${{varCor}}; margin-top:4px;">${{varSeta}} ${{Math.abs(variacao).toFixed(1)}}% vs anterior</div>` : ''}}
                </div>
                <div style="display:grid; grid-template-columns:1fr 1fr 1fr; gap:12px; margin-bottom:20px;">
                    <div style="text-align:center; padding:10px; border-radius:8px; background:var(--bg-page);">
                        <div style="font-size:11px; color:var(--text-muted);">Mínimo</div>
                        <div style="font-size:16px; font-weight:600;">${{fmt(minVal)}}</div>
                    </div>
                    <div style="text-align:center; padding:10px; border-radius:8px; background:var(--bg-page);">
                        <div style="font-size:11px; color:var(--text-muted);">Média</div>
                        <div style="font-size:16px; font-weight:600;">${{fmt(media)}}</div>
                    </div>
                    <div style="text-align:center; padding:10px; border-radius:8px; background:var(--bg-page);">
                        <div style="font-size:11px; color:var(--text-muted);">Máximo</div>
                        <div style="font-size:16px; font-weight:600;">${{fmt(maxVal)}}</div>
                    </div>
                </div>
                <div style="font-size:12px; font-weight:600; margin-bottom:8px; color:var(--text-muted);">ÚLTIMOS 10 REGISTROS</div>
                <table style="width:100%; border-collapse:collapse;">
                    <thead><tr>
                        <th style="text-align:left; padding:6px 8px; font-size:11px; color:var(--text-muted);">DATA</th>
                        <th style="text-align:left; padding:6px 8px; font-size:11px; color:var(--text-muted);">VALOR</th>
                    </tr></thead>
                    <tbody>${{ultimos}}</tbody>
                </table>`;
        }} catch(err) {{
            grafDiv.innerHTML = `<p style="color:#E63946">❌ ${{err.message}}</p>`;
            infoDiv.innerHTML = '';
        }}
    }}

    function voltarDashboard() {{
        document.getElementById('painel-detalhe').style.display  = 'none';
        ativarAba(localStorage.getItem('dashboard_tab') || 'ons');
    }}


    // ── Camada executiva da nova interface; usa os cards já processados ─────────
    function _cardPorTexto(texto, seletor) {{
        var cards = Array.from(document.querySelectorAll(seletor || '#grid-ons .card-item, #grid-ccee .card-item'));
        texto = texto.toLowerCase();
        return cards.find(function(card) {{
            return (card.querySelector('.card-title')?.textContent || '').toLowerCase().indexOf(texto) >= 0;
        }});
    }}
    function _valorCard(card) {{
        return card?.querySelector('.card-kpi-value')?.textContent.trim() || '—';
    }}
    function montarVisaoGeral() {{
        var cvu = document.querySelector('#grid-ons .cvu-consolidado');
        var pld = document.querySelector('#grid-ccee .pld-card') || _cardPorTexto('pld');
        var carga = _cardPorTexto('carga');
        var ear = _cardPorTexto('armazenamento') || _cardPorTexto('ear');
        var setValue = function(id, value) {{ var el = document.getElementById(id); if (el) el.textContent = value || '—'; }};
        setValue('overview-kpi-cvu', _valorCard(cvu));
        setValue('overview-kpi-pld', _valorCard(pld));
        setValue('overview-kpi-carga', _valorCard(carga));
        setValue('overview-kpi-ear', _valorCard(ear));
        if (!cvu) return;
        var source = cvu.querySelector('.source-link');
        var sourceTarget = document.getElementById('overview-detail-source');
        if (source && sourceTarget) sourceTarget.href = source.href;
        var kpi = cvu.querySelector('.card-kpi-value');
        setValue('overview-detail-usinas', kpi ? kpi.textContent.trim() : '—');
        var firstSummary = cvu.querySelector('.cvu-region-summary span');
        setValue('overview-detail-valor', firstSummary ? firstSummary.textContent.trim() : '—');
        var summary = cvu.querySelector('.cvu-region-summary-grid');
        var summaryTarget = document.getElementById('overview-cvu-summary');
        if (summaryTarget) summaryTarget.innerHTML = summary ? summary.outerHTML : '<div class="cvu-help">Resumo regional indisponível.</div>';
        var plotSource = cvu.querySelector('.plotly-lazy');
        var plotTarget = document.getElementById('overview-cvu-chart');
        if (plotSource && plotTarget) {{
            try {{
                var spec = JSON.parse(plotSource.dataset.spec || '{{}}');
                var layout = Object.assign({{}}, spec.layout || {{}}, {{height:405, margin:{{l:58,r:18,t:10,b:112}}, showlegend:false, responsive:true}});
                Plotly.react(plotTarget, spec.data || [], layout, {{responsive:true, displaylogo:false}});
            }} catch (e) {{
                plotTarget.innerHTML = '<p style="color:var(--text-muted);padding:30px;">Gráfico indisponível para visualização.</p>';
            }}
        }}
    }}
    function exportarPainel() {{
        ativarAba('ons');
        var primeiro = document.querySelector('#grid-ons .card-item');
        if (primeiro) primeiro.scrollIntoView({{behavior:'smooth', block:'start'}});
    }}
    setTimeout(montarVisaoGeral, 0);

    // Preferências locais e filtros visuais; não alteram dados, consultas ou cálculos existentes.
    var uxBuscaAtual = '';
    var uxFonteAtual = '';
    var uxSomenteFavoritos = false;
    function uxCards() {{ return Array.from(document.querySelectorAll('.card-item')); }}
    function uxAplicarBusca(valor) {{ uxBuscaAtual = (valor || '').toLowerCase().trim(); uxFiltrarCards(); }}
    function uxAplicarFonte(valor) {{ uxFonteAtual = (valor || '').toLowerCase(); uxFiltrarCards(); }}
    function uxFiltrarCards() {{
        var cards = uxCards();
        cards.forEach(function(card) {{
            var texto = (card.textContent || '').toLowerCase();
            var fonteOk = !uxFonteAtual || texto.indexOf(uxFonteAtual) >= 0;
            var buscaOk = !uxBuscaAtual || texto.indexOf(uxBuscaAtual) >= 0;
            var favoritoOk = !uxSomenteFavoritos || card.classList.contains('is-favorite');
            card.classList.toggle('ux-hidden', !(fonteOk && buscaOk && favoritoOk));
        }});
    }}
    function uxMostrarFavoritos() {{ uxSomenteFavoritos = !uxSomenteFavoritos; var b=document.getElementById('ux-favoritos'); b.classList.toggle('ux-btn-primary', uxSomenteFavoritos); b.setAttribute('aria-pressed', String(uxSomenteFavoritos)); uxFiltrarCards(); }}
    function uxLimparFiltros() {{ document.getElementById('ux-busca').value=''; document.getElementById('ux-fonte').value=''; uxBuscaAtual=''; uxFonteAtual=''; uxSomenteFavoritos=false; var b=document.getElementById('ux-favoritos'); b.classList.remove('ux-btn-primary'); b.setAttribute('aria-pressed','false'); uxFiltrarCards(); }}
    function uxAlternarDensidade(v) {{ document.body.classList.toggle('compact-mode', v === 'compacta'); localStorage.setItem('dashboard_density', v); }}
    function uxAtualizarVisual() {{ var b=event && event.currentTarget; if (b) {{ b.textContent='Visão atualizada'; setTimeout(function(){{b.textContent='Atualizar visão';}},1400); }} uxFiltrarCards(); }}
    function uxInstalarFavoritos() {{
        uxCards().forEach(function(card, idx) {{
            if (card.querySelector('.ux-fav-btn')) return;
            var btn=document.createElement('button'); btn.type='button'; btn.className='ux-btn ux-fav-btn'; btn.title='Fixar indicador'; btn.setAttribute('aria-label','Fixar indicador'); btn.textContent='☆';
            var key='dashboard_fav_'+idx; var saved=localStorage.getItem(key)==='1'; card.classList.toggle('is-favorite',saved); btn.textContent=saved?'★':'☆'; btn.setAttribute('aria-pressed',String(saved));
            btn.onclick=function(){{ var active=card.classList.toggle('is-favorite'); btn.textContent=active?'★':'☆'; btn.setAttribute('aria-pressed',String(active)); localStorage.setItem(key,active?'1':'0'); uxFiltrarCards(); }};
            var host=card.querySelector('.card-actions') || card.querySelector('.card-header'); if(host) host.appendChild(btn);
        }});
        var density=localStorage.getItem('dashboard_density') || 'normal'; var sel=document.getElementById('ux-densidade'); if(sel) sel.value=density; uxAlternarDensidade(density);
    }}
    setTimeout(uxInstalarFavoritos, 80);



    // ── Mapa meteorológico Windy incorporado em módulo isolado ─────────────────
    const WINDY_MAP_BASE = 'https://embed.windy.com/embed.html?type=map&location=coordinates&metricRain=default&metricTemp=default&metricWind=default&zoom=4&level=surface&lat=-14.2&lon=-51.9';
    function carregarMapaWindy(layer, model) {{
        var iframe = document.getElementById('windy-map');
        if (!iframe) return;
        iframe.src = WINDY_MAP_BASE + '&overlay=' + encodeURIComponent(layer || 'wind') + '&product=' + encodeURIComponent(model || 'ecmwf');
        var label = document.getElementById('windy-model-label');
        if (label) label.textContent = (model || 'ecmwf').toUpperCase();
    }}
    function alterarCamadaWindy(layer) {{
        var model = document.getElementById('windy-model')?.value || 'ecmwf';
        carregarMapaWindy(layer, model);
    }}
    function alterarModeloWindy(model) {{
        var layer = document.getElementById('windy-layer')?.value || 'wind';
        carregarMapaWindy(layer, model);
    }}
    setTimeout(function() {{ carregarMapaWindy('wind', 'ecmwf'); }}, 0);
</script></body></html>"""

with open('dashboard_ons_v2.html', 'w', encoding='utf-8') as f:
    f.write(html_template)

print("✅ Dashboard V2 gerado com sucesso!")
display(HTML(html_template))